# Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import yaml

# Point to the YOLO dataset root
ROOT = Path("/content/drive/MyDrive/isaac_scene1/yolo_patched")

assert ROOT.exists(), f"ROOT not found: {ROOT}"

# Try to load names from existing data.yaml first
existing_yaml = ROOT / "data.yaml"
assert existing_yaml.exists(), f"Missing data.yaml at: {existing_yaml}"

with open(existing_yaml, "r") as f:
    existing_data = yaml.safe_load(f)

# Load class names
raw_names = existing_data["names"]
if isinstance(raw_names, dict):
    names = {int(k): v for k, v in raw_names.items()}
else:
    names = {i: n for i, n in enumerate(raw_names)}

print(f"Loaded {len(names)} classes.")
print(names)

# Detect dataset layout
# Layout A: ROOT/images/{train,val,test} and ROOT/labels/{train,val,test}
layoutA = (ROOT / "images" / "train").exists() and (ROOT / "labels" / "train").exists()

# Layout B: ROOT/{train,val,test}/images and ROOT/{train,val,test}/labels
layoutB = (ROOT / "train" / "images").exists() and (ROOT / "train" / "labels").exists()

assert layoutA or layoutB, (
    "Couldn't detect splits. Expected either:\n"
    "A) ROOT/images/train + ROOT/labels/train\n"
    "or\n"
    "B) ROOT/train/images + ROOT/train/labels"
)

if layoutA:
    train_path = "images/train"
    val_path   = "images/val"
    test_path  = "images/test"
    print("Detected Layout A: ROOT/images/{train,val,test} and ROOT/labels/{train,val,test}")
else:
    train_path = "train/images"
    val_path   = "val/images"
    test_path  = "test/images"
    print("Detected Layout B: ROOT/{train,val,test}/images and ROOT/{train,val,test}/labels")

data = {
    "path": str(ROOT),
    "train": train_path,
    "val": val_path,
    "test": test_path,
    "names": names
}

OUT_YAML = Path("/content/isaac_correct.yaml")
OUT_YAML.write_text(yaml.safe_dump(data, sort_keys=False))

print("\nWrote:", OUT_YAML)
print(OUT_YAML.read_text())

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATA_YAML = "/content/isaac_correct.yaml"
best_pt = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene1/weights/best.pt"

# sanity checks
print("DATA_YAML exists:", Path(DATA_YAML).exists())
print("BEST_PT exists:", Path(best_pt).exists())

model = YOLO(best_pt)

# Clean test evaluation
test_res = model.val(
    data=DATA_YAML,
    split="test",
    imgsz=640,
    conf=0.25,
    iou=0.7
)

print(test_res)

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO

##Create the Learnable Patch

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
patch_size = (80, 80)

# Learnable patch initialized with random values
patch = torch.rand(3, *patch_size, device=device, requires_grad=True)

# Setup optimizer
optimizer = torch.optim.Adam([patch], lr=0.05)

print(f"Patch shape: {patch.shape} | Device: {device}")


## Apply Appearance Transformations

In [ ]:
import torchvision.transforms as T

appearance_transforms = T.Compose([
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.RandomPerspective(distortion_scale=0.5, p=0.5),
    T.GaussianBlur(3),
])

def apply_appearance_transforms(patch):
    return appearance_transforms(patch)



## EOT — Spatial + Projective Transformations

In [ ]:
!pip -q install kornia

import kornia

def apply_eot(patch):
    # Simulate different angles/scales
    affine = kornia.augmentation.RandomAffine(degrees=20, translate=0.1, scale=(0.8, 1.2))
    return affine(patch.unsqueeze(0)).squeeze(0)


## Add Patch to Image

In [ ]:
import torch
import random

def overlay_patch(image, patch, x_offset=None, y_offset=None):
    patched = image.clone()
    _, H_img, W_img = image.shape
    _, H_patch, W_patch = patch.shape

    # Calculate valid bounds
    max_y = H_img - H_patch
    max_x = W_img - W_patch

    if max_y < 0 or max_x < 0:
        raise ValueError("Patch is larger than the image.")

    # If no offsets are given, random position
    if x_offset is None:
        x_offset = random.randint(0, max_x)
    else:
        x_offset = min(max(x_offset, 0), max_x)

    if y_offset is None:
        y_offset = random.randint(0, max_y)
    else:
        y_offset = min(max(y_offset, 0), max_y)

    # Apply the patch
    patched[:, y_offset:y_offset + H_patch, x_offset:x_offset + W_patch] = patch
    return patched



# Load trained YOLOv8 model


In [ ]:
YOLO_MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene1/weights/best.pt"  # <- your trained model


In [ ]:
from ultralytics import YOLO

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load trained YOLOv8 model
model = YOLO(YOLO_MODEL_PATH)
model.fuse()

model.model.to(device)
model.to(device)


In [ ]:
def overlay_patch(image, patch, x_offset=0, y_offset=0):
    patched_image = image.clone()
    _, H, W = image.shape
    ph, pw = patch.shape[1:]

    # Ensure patch fits
    if y_offset + ph > H or x_offset + pw > W:
        raise ValueError("Patch placement is outside image bounds.")

    patched_image[:, y_offset:y_offset+ph, x_offset:x_offset+pw] = patch
    return patched_image


# Optimizing Patch on a folder (e.g. billboard01)

In [ ]:
from torch.utils.data import DataLoader
import torchvision.transforms as T
import os
from glob import glob

image_dir = "/content/drive/MyDrive/isaac_scene1/yolo_filtered/images/test"
label_dir = "/content/drive/MyDrive/isaac_scene1/yolo_filtered/labels/test"

assert os.path.exists(image_dir), f"Image directory not found: {image_dir}"
assert os.path.exists(label_dir), f"Label directory not found: {label_dir}"

image_paths = sorted(glob(os.path.join(image_dir, "*.png")))
label_paths = sorted(glob(os.path.join(label_dir, "*.txt")))

print(f"Found {len(image_paths)} test images")
print(f"Found {len(label_paths)} test labels")

In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms

# Load all image paths from the directory
image_paths = [os.path.join(image_dir, fname) for fname in os.listdir(image_dir) if fname.endswith(".jpg") or fname.endswith(".png")]

Patch optimized over multiple random scales to improve robustness to viewpoint distance.

In [ ]:
# =========================
# 3-seed patch optimization + 5-scale robustness (ALL 5 scales per image per step)
# Uses a SINGLE backward per image (sum of 5-scale losses) to avoid "backward twice" error.
# Saves: bill1a.pt, bill1b.pt, bill1c.pt and best_bill1.pt
# =========================

import os
import random
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image

# ---------- Config ----------
SEEDS = [0, 1, 2]
SEED_TAGS = {0: "a", 1: "b", 2: "c"}   # bill1a/b/c naming

SAVE_DIR = "./patch_outputs"
# SAVE_DIR = "/content/drive/MyDrive/yolo_patch/IsaacSim/patch_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

K = 400
TH = 0.25
LAMBDA_TH = 1.0
LR = 0.03
STEPS = 200

# Anchor position for BASE patch size
X_OFFSET = 300
Y_OFFSET = 300

# ✅ Your requested 5 scales (zoom in/out robustness)
# NOTE: Python uses "." for decimals, so 1,00 -> 1.00
SCALES = [0.85, 1.00, 1.20, 1.45, 1.80]

# Optional: limit images per step to reduce GPU load (None = all)
# If GPU crashes, set to 8 or 16.
IMAGES_PER_STEP = None  # e.g. 16

# ---------- Setup ----------
device = next(model.model.parameters()).device
print("Model device:", device)

nc = int(getattr(model.model, "nc", 80))
print("Model nc:", nc)

# Recommended stability
model.model.eval()
image_paths = sorted(image_paths)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(False)

def get_any_confidence(preds, nc: int) -> torch.Tensor:
    """
    Handles YOLOv8 outputs that may be (N,D) or (D,N) or (1,N,D).
    D = 4 + nc OR 5 + nc.
    Returns conf_any: (N,) = max class score per box (with obj if present).
    """
    if preds.dim() == 3:
        preds = preds[0]

    # If transposed (D,N), convert to (N,D)
    if preds.shape[0] in (4 + nc, 5 + nc) and preds.shape[1] not in (4 + nc, 5 + nc):
        preds = preds.t()

    D = preds.shape[1]
    if D == 5 + nc:
        obj = preds[:, 4]
        cls_scores = preds[:, 5:]
        return obj * cls_scores.max(dim=1).values

    if D == 4 + nc:
        cls_scores = preds[:, 4:]
        return cls_scores.max(dim=1).values

    raise ValueError(f"Unexpected preds shape {preds.shape}. Expected D={5+nc} or D={4+nc}.")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((640, 640)),
])

# Patch base must already exist in your notebook/script
patch_base = patch.to(device)
_, BASE_H, BASE_W = patch_base.shape

def resize_patch(p: torch.Tensor, scale: float) -> torch.Tensor:
    """Resize patch tensor (C,H,W) by scale using torch ops (keeps grads)."""
    C, H, W = p.shape
    new_h = max(1, int(round(H * scale)))
    new_w = max(1, int(round(W * scale)))
    return F.interpolate(
        p.unsqueeze(0),
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

def center_preserving_offsets(x_offset: int, y_offset: int, new_h: int, new_w: int):
    """Adjust top-left offsets so patch stays roughly centered when size changes."""
    x = int(round(x_offset - (new_w - BASE_W) / 2))
    y = int(round(y_offset - (new_h - BASE_H) / 2))
    return x, y

# Store results for best selection
seed_results = {}  # seed -> dict(best_loss, best_patch_cpu, save_path)

for seed in SEEDS:
    tag = SEED_TAGS[seed]
    save_path = os.path.join(SAVE_DIR, f"bill1{tag}.pt")

    print("\n" + "=" * 60)
    print(f"Running seed = {seed}  -> will save: {save_path}")
    print("=" * 60)

    set_seed(seed)

    # fresh patch + optimizer per seed
    patch4 = torch.rand_like(patch_base, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([patch4], lr=LR)

    best_loss = float("inf")
    best_patch4_cpu = patch4.detach().clone().cpu()

    for step in range(STEPS):
        optimizer.zero_grad(set_to_none=True)

        # Optionally sample subset images (speed/stability)
        if IMAGES_PER_STEP is None or IMAGES_PER_STEP >= len(image_paths):
            step_paths = image_paths
        else:
            step_paths = random.sample(image_paths, k=IMAGES_PER_STEP)

        # We do ALL 5 scales per image
        denom = len(step_paths) * len(SCALES)

        total_loss_val = 0.0
        total_boxes_over_th = 0

        for img_path in step_paths:
            img = Image.open(img_path).convert("RGB")
            image_tensor = transform(img).to(device)

            # ✅ Sum 5-scale losses, then backward ONCE per image
            loss_img = 0.0

            for sc in SCALES:
                # IMPORTANT: build fresh patch transforms each scale (avoids "backward twice" on same graph)
                t_patch = apply_appearance_transforms(patch4)
                t_patch = apply_eot(t_patch)

                # scale patch
                s_patch = resize_patch(t_patch, sc).clamp(0, 1)
                _, ph, pw = s_patch.shape
                x_off, y_off = center_preserving_offsets(X_OFFSET, Y_OFFSET, ph, pw)

                # overlay
                patched_image = overlay_patch(
                    image_tensor, s_patch, x_offset=x_off, y_offset=y_off
                )
                patched_input = patched_image.unsqueeze(0)

                # Forward raw head (your working call)
                preds = model.model(patched_input)[0]

                # conf
                conf_any = get_any_confidence(preds, nc)

                # losses
                topk = torch.topk(conf_any, k=min(K, conf_any.numel())).values
                loss_topk = -torch.log(topk + 1e-6).sum()
                loss_th = -torch.relu(conf_any - TH).sum()
                loss = loss_topk + LAMBDA_TH * loss_th

                loss_img = loss_img + loss

                # logging (detach only)
                total_loss_val += loss.detach().item()
                total_boxes_over_th += (conf_any > TH).sum().item()

            # ✅ single backward for this image across all 5 scales
            (loss_img / max(1, denom)).backward()

        optimizer.step()
        patch4.data.clamp_(0, 1)

        avg_loss = total_loss_val / max(1, denom)
        avg_boxes_over_th = total_boxes_over_th / max(1, denom)

        # Track best patch by lowest avg_loss
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_patch4_cpu = patch4.detach().clone().cpu()

        if step % 10 == 0:
            print(
                f"[seed {seed}] Step {step:3d} | Loss: {avg_loss:.4f} | "
                f"Avg boxes>TH({TH}): {avg_boxes_over_th:.2f} | Scales: {SCALES}"
            )

    # Save the best patch for this seed
    torch.save(best_patch4_cpu, save_path)
    print(f"Saved seed {seed} best patch to: {save_path}")

    seed_results[seed] = {
        "best_loss": best_loss,
        "best_patch_cpu": best_patch4_cpu,
        "save_path": save_path,
    }

# Choose overall best patch by lowest best_loss
best_seed = min(seed_results.keys(), key=lambda s: seed_results[s]["best_loss"])
best_patch_overall = seed_results[best_seed]["best_patch_cpu"]
best_path = os.path.join(SAVE_DIR, "best_bill1.pt")
torch.save(best_patch_overall, best_path)

print("\n" + "#" * 70)
print(f"Best seed = {best_seed}  with best_loss = {seed_results[best_seed]['best_loss']:.6f}")
print(f"Saved overall best patch to: {best_path}")
print("#" * 70)


In [ ]:
import os
import torch
import torchvision.utils as vutils

# ----------------------------
# Config
# ----------------------------
# Where patches were saved by training (local runtime)
SRC_DIR = "./patch_outputs"

# Where to store on Google Drive (persistent)
OUT_DIR = "/content/drive/MyDrive/isaac_scene1/yolo_patch/IsaacSim/patch_outputs_saved"
os.makedirs(OUT_DIR, exist_ok=True)

# Desired names on Drive
name_map = {
    "isaac_patch_seed0": "bill1a.pt",
    "isaac_patch_seed1": "bill1b.pt",
    "isaac_patch_seed2": "bill1c.pt",
    "isaac_patch_best":  "best_bill1.pt",
}

# ----------------------------
# Load + Save (png + pt) to Drive
# ----------------------------
for out_name, src_file in name_map.items():
    src_path = os.path.join(SRC_DIR, src_file)
    if not os.path.exists(src_path):
        print(f"❌ Missing: {src_path} (skip)")
        continue

    p = torch.load(src_path, map_location="cpu").detach().clone().clamp(0, 1).cpu()

    png_path = os.path.join(OUT_DIR, f"{out_name}.png")
    pt_path  = os.path.join(OUT_DIR, f"{out_name}.pt")

    vutils.save_image(p, png_path)
    torch.save(p, pt_path)

    print(f"✅ Saved: {png_path}")
    print(f"✅ Saved: {pt_path}")

# ----------------------------
# Optional: simple alias for demos (same as best)
# ----------------------------
best_src = os.path.join(SRC_DIR, "best_bill1.pt")
if os.path.exists(best_src):
    best_patch = torch.load(best_src, map_location="cpu").detach().clone().clamp(0, 1).cpu()

    vutils.save_image(best_patch, os.path.join(OUT_DIR, "isaac_patch.png"))
    torch.save(best_patch, os.path.join(OUT_DIR, "isaac_patch.pt"))

    print("⭐ Saved aliases: isaac_patch.png and isaac_patch.pt (== isaac_patch_best)")
else:
    print(f"❌ Missing best patch: {best_src}")

print("\nDone. Output folder:", OUT_DIR)


In [ ]:
from ultralytics import YOLO

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
model = YOLO(MODEL_PATH)

print("Model loaded successfully")

In [ ]:
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from torchvision import transforms

# ----------------------------
# Config
# ----------------------------
PATCH_PT = "/content/drive/MyDrive/isaac_scene1/yolo_patch/IsaacSim/patch_outputs_saved/isaac_patch.pt"
OUT_DIR = "/content/drive/MyDrive/isaac_scene1/yolo_patch/patch_eval_sizes"
os.makedirs(OUT_DIR, exist_ok=True)

DEMO_IMAGE_PATH = image_paths[10]   # change if you want

# anchor offsets for BASE patch size
X_OFFSET = 300
Y_OFFSET = 300

# your requested sizes
SCALES = [0.85, 1.00, 1.20, 1.45, 1.80]

TH = 0.25  # evaluation threshold (same as training)

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((640, 640)),
])

# ----------------------------
# Helpers
# ----------------------------
def get_any_confidence(preds, nc: int) -> torch.Tensor:
    """Works with YOLOv8-ish head outputs (N,D) or (D,N) or (1,N,D)."""
    if preds.dim() == 3:
        preds = preds[0]

    # If transposed (D,N) -> (N,D)
    if preds.shape[0] in (4 + nc, 5 + nc) and preds.shape[1] not in (4 + nc, 5 + nc):
        preds = preds.t()

    D = preds.shape[1]
    if D == 5 + nc:
        obj = preds[:, 4]
        cls_scores = preds[:, 5:]
        return obj * cls_scores.max(dim=1).values
    if D == 4 + nc:
        cls_scores = preds[:, 4:]
        return cls_scores.max(dim=1).values

    raise ValueError(f"Unexpected preds shape {preds.shape}. Expected D={5+nc} or D={4+nc}.")

def resize_patch(p: torch.Tensor, scale: float) -> torch.Tensor:
    """Resize patch (C,H,W) by scale using torch ops."""
    C, H, W = p.shape
    new_h = max(1, int(round(H * scale)))
    new_w = max(1, int(round(W * scale)))
    return F.interpolate(
        p.unsqueeze(0),
        size=(new_h, new_w),
        mode="bilinear",
        align_corners=False
    ).squeeze(0)

def center_preserving_offsets(x_offset: int, y_offset: int, base_h: int, base_w: int, new_h: int, new_w: int):
    """Keep patch centered around the same place when size changes."""
    x = int(round(x_offset - (new_w - base_w) / 2))
    y = int(round(y_offset - (new_h - base_h) / 2))
    return x, y

def safe_det_count(result0):
    """Try to count detections from Ultralytics Results (if present)."""
    try:
        if hasattr(result0, "boxes") and result0.boxes is not None:
            return int(result0.boxes.shape[0]) if hasattr(result0.boxes, "shape") else int(len(result0.boxes))
    except Exception:
        pass
    return None

# ----------------------------
# Load model info + patch + image
# ----------------------------
assert os.path.exists(PATCH_PT), f"Patch not found: {PATCH_PT}"
assert os.path.exists(DEMO_IMAGE_PATH), f"Image not found: {DEMO_IMAGE_PATH}"

device = next(model.model.parameters()).device
nc = int(getattr(model.model, "nc", 80))

patch_cpu = torch.load(PATCH_PT, map_location="cpu").clamp(0, 1)
base_c, base_h, base_w = patch_cpu.shape

img = Image.open(DEMO_IMAGE_PATH).convert("RGB")
image_tensor = demo_transform(img).to(device)

print("Device:", device)
print("nc:", nc)
print("Patch:", PATCH_PT, "| patch:", tuple(patch_cpu.shape))
print("Demo:", DEMO_IMAGE_PATH, "| image_tensor:", tuple(image_tensor.shape))
print("OUT_DIR:", OUT_DIR)

# ----------------------------
# Clean (reference)
# ----------------------------
with torch.no_grad():
    clean_np = image_tensor.clamp(0, 1).detach().cpu().permute(1, 2, 0).numpy()

    # YOLO plotted detections (for visualization)
    model.model.eval()
    clean_det = model(image_tensor.unsqueeze(0), verbose=False)
    clean_plot_bgr = clean_det[0].plot()
    clean_plot_rgb = cv2.cvtColor(clean_plot_bgr, cv2.COLOR_BGR2RGB)

    # raw-head evaluation (boxes>TH)
    preds_clean = model.model(image_tensor.unsqueeze(0))[0]
    conf_clean = get_any_confidence(preds_clean, nc)
    clean_boxes_over_th = int((conf_clean > TH).sum().item())

    clean_det_count = safe_det_count(clean_det[0])

# Save clean images
clean_nodet_path = os.path.join(OUT_DIR, "clean_nodet.png")
clean_withdet_path = os.path.join(OUT_DIR, "clean_withdet.png")
Image.fromarray((clean_np * 255).astype("uint8")).save(clean_nodet_path)
Image.fromarray(clean_plot_rgb).save(clean_withdet_path)

print(f"Saved clean: {clean_nodet_path}")
print(f"Saved clean: {clean_withdet_path}")

# ----------------------------
# Evaluate each scale
# ----------------------------
rows = []
patched_nodet_imgs = []
patched_withdet_imgs = []

with torch.no_grad():
    patch_gpu_base = patch_cpu.to(device)

    for sc in SCALES:
        # scale patch
        p_sc = resize_patch(patch_gpu_base, sc).clamp(0, 1)
        _, ph, pw = p_sc.shape

        # center-preserving offsets
        x_off, y_off = center_preserving_offsets(X_OFFSET, Y_OFFSET, base_h, base_w, ph, pw)

        # overlay
        patched = overlay_patch(
            image_tensor.clone(),
            p_sc,
            x_offset=x_off,
            y_offset=y_off
        )

        # raw-head metric (boxes > TH)
        preds = model.model(patched.unsqueeze(0))[0]
        conf_any = get_any_confidence(preds, nc)
        boxes_over_th = int((conf_any > TH).sum().item())
        top_conf = float(conf_any.max().item()) if conf_any.numel() > 0 else 0.0

        # YOLO plotted detections
        det = model(patched.unsqueeze(0), verbose=False)
        det_count = safe_det_count(det[0])

        plot_bgr = det[0].plot()
        plot_rgb = cv2.cvtColor(plot_bgr, cv2.COLOR_BGR2RGB)

        # save images
        patched_np = patched.clamp(0, 1).detach().cpu().permute(1, 2, 0).numpy()

        nodet_path = os.path.join(OUT_DIR, f"patched_scale_{sc:.2f}_nodet.png")
        withdet_path = os.path.join(OUT_DIR, f"patched_scale_{sc:.2f}_withdet.png")
        Image.fromarray((patched_np * 255).astype("uint8")).save(nodet_path)
        Image.fromarray(plot_rgb).save(withdet_path)

        patched_nodet_imgs.append(Image.open(nodet_path).convert("RGB"))
        patched_withdet_imgs.append(Image.open(withdet_path).convert("RGB"))

        rows.append({
            "scale": sc,
            "patch_hw": (ph, pw),
            "offset_xy": (x_off, y_off),
            "raw_boxes>TH": boxes_over_th,
            "raw_max_conf": round(top_conf, 4),
            "yolo_det_count": det_count,
            "saved_nodet": nodet_path,
            "saved_withdet": withdet_path
        })

# ----------------------------
# Print evaluation table
# ----------------------------
print("\n=== CLEAN reference ===")
print(f"raw_boxes>TH({TH}) = {clean_boxes_over_th} | yolo_det_count = {clean_det_count}")

print("\n=== PATCHED by scale ===")
for r in rows:
    print(
        f"scale={r['scale']:.2f} | patch={r['patch_hw']} | offset={r['offset_xy']} | "
        f"raw_boxes>TH={r['raw_boxes>TH']} | raw_max_conf={r['raw_max_conf']} | yolo_det_count={r['yolo_det_count']}"
    )

# ----------------------------
# Visualize: clean + all patched
# ----------------------------
# Row 1: no det (clean + 5 scales)
plt.figure(figsize=(20, 7))
plt.subplot(2, 6, 1)
plt.imshow(clean_np)
plt.title("Clean (no det)")
plt.axis("off")

for i, sc in enumerate(SCALES, start=2):
    plt.subplot(2, 6, i)
    plt.imshow(patched_nodet_imgs[i-2])
    plt.title(f"Scale {SCALES[i-2]:.2f} (no det)")
    plt.axis("off")

# Row 2: with det (clean + 5 scales)
plt.subplot(2, 6, 7)
plt.imshow(clean_plot_rgb)
plt.title("Clean (with det)")
plt.axis("off")

for i, sc in enumerate(SCALES, start=8):
    plt.subplot(2, 6, i)
    plt.imshow(patched_withdet_imgs[i-8])
    plt.title(f"Scale {SCALES[i-8]:.2f} (with det)")
    plt.axis("off")

plt.tight_layout()
plt.show()


## Create a dataset with patched images

In [ ]:
# ===========================
# CREATE ONE PATCHED DATASET
# ===========================
import os, glob, shutil
import torch
from PIL import Image
from torchvision import transforms
from torchvision.utils import save_image
from tqdm import tqdm

# ---- SETTINGS ----
PATCH_PATH   = "/content/drive/MyDrive/yolo_patch/Billboard1/patches/best_bill1.pt"
DATASET_ROOT = "/content/drive/MyDrive/isaac_scene1/clean/yolo_filtered"
OUT_ROOT     = "/content/drive/MyDrive/isaac_scene1/transf_Patch1"

X_OFFSET, Y_OFFSET = 400, 270
IMGSZ = 640

# ---- INPUT DIRS ----
images_dir = os.path.normpath(os.path.join(DATASET_ROOT, "images", "test"))
labels_dir = os.path.normpath(os.path.join(DATASET_ROOT, "labels", "test"))

# ---- OUTPUT DIRS ----
out_images_dir = os.path.normpath(os.path.join(OUT_ROOT, "images", "test"))
out_labels_dir = os.path.normpath(os.path.join(OUT_ROOT, "labels", "test"))

# Clean output first
if os.path.exists(OUT_ROOT):
    shutil.rmtree(OUT_ROOT)
os.makedirs(out_images_dir, exist_ok=True)
os.makedirs(out_labels_dir, exist_ok=True)

# ---- DEBUG ----
print("PATCH_PATH:", PATCH_PATH)
print("DATASET_ROOT:", DATASET_ROOT)
print("images_dir:", images_dir)
print("labels_dir:", labels_dir)
print("OUT_ROOT:", OUT_ROOT)

print("PATCH_PATH exists:", os.path.exists(PATCH_PATH))
print("DATASET_ROOT exists:", os.path.exists(DATASET_ROOT))
print("images_dir exists:", os.path.exists(images_dir))
print("labels_dir exists:", os.path.exists(labels_dir))

if not os.path.exists(PATCH_PATH):
    raise FileNotFoundError(f"Patch file not found: {PATCH_PATH}")

if not os.path.exists(DATASET_ROOT):
    raise FileNotFoundError(f"Dataset root not found: {DATASET_ROOT}")

if not os.path.exists(images_dir):
    raise FileNotFoundError(f"Images directory not found: {images_dir}")

if not os.path.exists(labels_dir):
    print(f"Warning: labels directory not found: {labels_dir}")
    print("Empty label files will be created.")

# ---- DEVICE ----
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- LOAD PATCH ----
patch = torch.load(PATCH_PATH, map_location="cpu")

if isinstance(patch, (list, tuple)):
    patch = patch[0]
if patch.ndim == 4:
    patch = patch[0]

patch = patch.float()
if patch.max().item() > 1.0:
    patch = patch / 255.0
patch = patch.to(device).clamp(0, 1)

print("Loaded patch shape:", tuple(patch.shape))

# ---- TRANSFORM ----
transform = transforms.Compose([
    transforms.Resize((IMGSZ, IMGSZ)),
    transforms.ToTensor(),
])

def overlay_patch(image_tensor, patch_tensor, x_offset=X_OFFSET, y_offset=Y_OFFSET):
    patched = image_tensor.clone()
    _, H, W = image_tensor.shape
    _, ph, pw = patch_tensor.shape

    if x_offset < 0 or y_offset < 0 or x_offset + pw > W or y_offset + ph > H:
        raise ValueError(
            f"Patch out of bounds: img=({H},{W}) patch=({ph},{pw}) offset=({x_offset},{y_offset})"
        )

    patched[:, y_offset:y_offset+ph, x_offset:x_offset+pw] = patch_tensor
    return patched

# ---- FIND IMAGES ----
exts = ("*.png", "*.jpg", "*.jpeg", "*.PNG", "*.JPG", "*.JPEG")
img_paths = []
for e in exts:
    img_paths.extend(glob.glob(os.path.join(images_dir, e)))

# Normalize and keep only real files inside the intended folder
img_paths = [os.path.normpath(p) for p in img_paths]
img_paths = [p for p in img_paths if os.path.isfile(p)]
img_paths = [p for p in img_paths if p.startswith(images_dir + os.sep)]
img_paths = sorted(set(img_paths))

print(f"Found {len(img_paths)} valid images")

if len(img_paths) == 0:
    raise RuntimeError(f"No valid images found in {images_dir}")

# Optional: inspect collected paths
print("\nFirst 10 image paths:")
for p in img_paths[:10]:
    print(p)

# ---- PROCESS ----
total = 0
failed = []

with torch.no_grad():
    for img_path in tqdm(img_paths, desc="Patching images"):
        try:
            img_name = os.path.basename(img_path)
            stem, ext = os.path.splitext(img_name)

            img = Image.open(img_path).convert("RGB")
            img_tensor = transform(img).to(device)

            patched_tensor = overlay_patch(img_tensor, patch, X_OFFSET, Y_OFFSET)

            out_img_path = os.path.join(out_images_dir, img_name)
            save_image(patched_tensor.detach().cpu(), out_img_path)

            src_label = os.path.join(labels_dir, stem + ".txt")
            dst_label = os.path.join(out_labels_dir, stem + ".txt")

            if os.path.exists(src_label):
                shutil.copy(src_label, dst_label)
            else:
                open(dst_label, "w").close()

            total += 1

        except Exception as e:
            failed.append((img_path, str(e)))

print("\n✅ Patched dataset created!")
print("Output root:", OUT_ROOT)
print("Images folder:", out_images_dir)
print("Labels folder:", out_labels_dir)
print("Total images saved:", total)
print("Failed images:", len(failed))
print("Patch offsets used:", (X_OFFSET, Y_OFFSET))

if failed:
    print("\nFirst few failures:")
    for path, err in failed[:10]:
        print("-", path, "->", err)

In [ ]:
# =========================
# RUN evaluation_ab.py ON ONE DATASET
# by passing the same root as clean and patched
# =========================
import os
import sys
import json
import subprocess
from pathlib import Path

EVAL_SCRIPT = Path("/content/evaluation_ab.py")
MODEL_PATH = Path("/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt")
DATASET_ROOT = Path("/content/drive/MyDrive/isaac_scene1/transf_Patch1")

SPLIT = "test"   # train / val / test
OUT_ROOT = Path("/content/drive/MyDrive/isaac_eval_ab_runs")
RUN_NAME = f"single_dataset_{SPLIT}_via_eval_ab"

IMGSZ = 640
CONF_MIN = 0.001
IOU = 0.6
THRESHOLDS = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5
DET_K = "1,5,10,20,50"
MAX_DET = 1000

def must_exist(p, what="path"):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

must_exist(EVAL_SCRIPT, "evaluation_ab.py")
must_exist(MODEL_PATH, "model")
must_exist(DATASET_ROOT, "dataset root")
must_exist(DATASET_ROOT / "images" / SPLIT, f"images/{SPLIT}")
must_exist(DATASET_ROOT / "labels" / SPLIT, f"labels/{SPLIT}")

OUT_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(EVAL_SCRIPT),
    "--model", str(MODEL_PATH),
    "--clean_root", str("/content/drive/MyDrive/isaac_scene1/clean/yolo_filtered"),
    "--patched_root", str(DATASET_ROOT),
    "--out_root", str(OUT_ROOT),
    "--run_name", RUN_NAME,
    "--split", SPLIT,
    "--imgsz", str(IMGSZ),
    "--conf_min", str(CONF_MIN),
    "--iou", str(IOU),
    "--thresholds", THRESHOLDS,
    "--gt_iou_match", str(GT_IOU_MATCH),
    "--det_k", DET_K,
    "--max_det", str(MAX_DET),
]

print("[RUNNING]")
print(" ".join(cmd))

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = ""

proc = subprocess.run(cmd, env=env, text=True, capture_output=True)

print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"evaluation_ab.py failed with rc={proc.returncode}")

run_dir = OUT_ROOT / RUN_NAME
combined_path = run_dir / "combined_metrics.json"
summary_path = run_dir / "summary.json"

must_exist(combined_path, "combined_metrics.json")
must_exist(summary_path, "summary.json")

summary = json.loads(summary_path.read_text())

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\nSaved:")
print("Run dir:", run_dir)
print("Combined:", combined_path)
print("Summary:", summary_path)

In [ ]:
import os

DATASET_ROOT = None
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "yolo_filtered" in dirs:
        DATASET_ROOT = os.path.join(root, "yolo_filtered")
        break

print("Detected DATASET_ROOT:", DATASET_ROOT)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# =========================
# EDIT THESE PATHS
# =========================
MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
IMAGE_PATH = "/content/drive/MyDrive/isaac_scene1_patched/patched_lightning/rgb_0067.png"

# Optional
CONF = 0.25
IOU = 0.6
IMGSZ = 640
SAVE_PATH = "/content/detection_vis.png"

# =========================
# LOAD MODEL + IMAGE
# =========================
model = YOLO(MODEL_PATH)

img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Could not read image: {IMAGE_PATH}")

# =========================
# RUN DETECTION
# =========================
result = model.predict(
    source=img_bgr,
    conf=CONF,
    iou=IOU,
    imgsz=IMGSZ,
    verbose=False
)[0]

# =========================
# PRINT DETECTION SUMMARY
# =========================
names = model.names
boxes = result.boxes

print(f"Image: {Path(IMAGE_PATH).name}")
print(f"Total detections: {0 if boxes is None else len(boxes)}")
print("-" * 50)

if boxes is not None and len(boxes) > 0:
    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()
    clss = boxes.cls.cpu().numpy().astype(int)

    for i, (box, conf, cls_id) in enumerate(zip(xyxy, confs, clss), 1):
        cls_name = names[cls_id] if isinstance(names, (list, tuple)) else names[int(cls_id)]
        x1, y1, x2, y2 = map(int, box)
        print(f"{i:02d}. {cls_name:<20} conf={conf:.3f} box=({x1}, {y1}, {x2}, {y2})")
else:
    print("No detections.")

# =========================
# VISUALIZE DETECTIONS
# =========================
vis_bgr = result.plot()   # Ultralytics draws boxes + labels

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis_bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()

# =========================
# SAVE OUTPUT
# =========================
cv2.imwrite(SAVE_PATH, vis_bgr)
print(f"\nSaved visualization to: {SAVE_PATH}")

In [ ]:
%%writefile /content/evaluation_ab.py
import argparse, json, re, subprocess, sys
from pathlib import Path
import numpy as np
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ----------------- utils -----------------
def run(cmd, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            sys.stdout.write(line)
            f.write(line)
        rc = p.wait()
    return rc


def percentile(arr, p):
    if arr is None or len(arr) == 0:
        return None
    return float(np.percentile(np.asarray(arr, dtype=np.float32), p))


def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p


def model_names_as_list(model: YOLO):
    names = getattr(model, "names", None)
    if names is None:
        raise ValueError("Model has no .names; cannot build dataset YAML.")

    if isinstance(names, dict):
        max_k = max(int(k) for k in names.keys())
        out = [None] * (max_k + 1)
        for k, v in names.items():
            out[int(k)] = str(v)
        for i, v in enumerate(out):
            if v is None:
                out[i] = f"class_{i}"
        return out

    if isinstance(names, (list, tuple)):
        return [str(x) for x in names]

    raise ValueError(f"Unsupported model.names type: {type(names)}")


def read_dataset_yaml(yaml_path: Path):
    txt = yaml_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    d = {"path": None, "train": None, "val": None, "test": None}
    for ln in txt:
        ln = ln.strip()
        if not ln or ln.startswith("#"):
            continue
        for k in ("path", "train", "val", "test"):
            if ln.startswith(k + ":"):
                d[k] = ln.split(":", 1)[1].strip()
    return d


def _has_images(dir_path: Path) -> bool:
    if not dir_path.exists() or not dir_path.is_dir():
        return False
    for x in dir_path.iterdir():
        if x.is_file() and x.suffix.lower() in IMG_EXTS:
            return True
    return False


def get_image_dir_from_yaml(yaml_path: Path) -> Path:
    y = read_dataset_yaml(yaml_path)
    root = y.get("path")
    if not root:
        raise ValueError(f"Could not parse 'path:' from {yaml_path}")

    rel = y.get("test") or y.get("val") or y.get("train")
    if not rel:
        raise ValueError(f"Could not parse test/val/train from {yaml_path}")

    base = (Path(root) / rel).resolve()

    if _has_images(base):
        return base

    for sub in ("test", "val", "train"):
        cand = base / sub
        if _has_images(cand):
            return cand

    if base.name != "images":
        cand = (Path(root) / "images").resolve()
        if _has_images(cand):
            return cand
        for sub in ("test", "val", "train"):
            cand2 = cand / sub
            if _has_images(cand2):
                return cand2

    return base


def get_labels_dir_for_images(img_dir: Path) -> Path:
    parts = list(img_dir.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        return Path(*parts)
    return img_dir.parent / "labels"


def write_yaml(out_yaml: Path, root: Path, test_rel: str, names):
    out_yaml.parent.mkdir(parents=True, exist_ok=True)
    content = (
        f"path: {root}\n"
        f"train: {test_rel}\n"
        f"val: {test_rel}\n"
        f"test: {test_rel}\n"
        f"\nnc: {len(names)}\n"
        f"names: {json.dumps(names)}\n"
    )
    out_yaml.write_text(content, encoding="utf-8")


def parse_yolo_val_metrics_from_log(log_path: Path):
    pat_all = re.compile(
        r"^\s*all\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$"
    )
    per_class = {}
    pat_cls = re.compile(
        r"^\s*([A-Za-z0-9 _-]+)\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$"
    )

    images = instances = None
    P = R = mAP50 = mAP5095 = None

    for ln in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        m = pat_all.match(ln)
        if m:
            images = int(m.group(1))
            instances = int(m.group(2))
            P = float(m.group(3))
            R = float(m.group(4))
            mAP50 = float(m.group(5))
            mAP5095 = float(m.group(6))

        mc = pat_cls.match(ln)
        if mc:
            cls = mc.group(1).strip()
            if cls != "all":
                per_class[cls] = {
                    "images": int(mc.group(2)),
                    "instances": int(mc.group(3)),
                    "precision": float(mc.group(4)),
                    "recall": float(mc.group(5)),
                    "mAP50": float(mc.group(6)),
                    "mAP50-95": float(mc.group(7)),
                }

    return {
        "images": images,
        "instances": instances,
        "precision": P,
        "recall": R,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "per_class": per_class,
        "log_path": str(log_path),
    }


# ----------------- A) detection density -----------------
def compute_detection_density_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, det_k_list, max_det: int):
    img_dir = get_image_dir_from_yaml(yaml_path)
    names = model_names_as_list(model)

    det_counts = []
    max_conf_any = []
    has_ge_k = {str(k): 0 for k in det_k_list}
    has_any_conf_ge = {str(t): 0 for t in thresholds}
    total_dets_conf_ge = {str(t): 0 for t in thresholds}
    dets_per_img_conf_ge = {str(t): [] for t in thresholds}
    class_hist = {}

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
        max_det=max_det,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        boxes = r.boxes
        n = 0 if boxes is None else len(boxes)
        det_counts.append(n)

        if n == 0:
            max_conf_any.append(0.0)
            for t in thresholds:
                dets_per_img_conf_ge[str(t)].append(0)
        else:
            confs = boxes.conf.detach().cpu().numpy().astype(np.float32)
            max_conf_any.append(float(confs.max()))
            clss = boxes.cls.detach().cpu().numpy().astype(int)

            for c in clss:
                k = names[c] if (0 <= c < len(names)) else str(int(c))
                class_hist[k] = class_hist.get(k, 0) + 1

            for t in thresholds:
                n_ge = int((confs >= t).sum())
                total_dets_conf_ge[str(t)] += n_ge
                dets_per_img_conf_ge[str(t)].append(n_ge)

        for k in det_k_list:
            if n >= k:
                has_ge_k[str(k)] += 1

        for t in thresholds:
            if max_conf_any[-1] >= t:
                has_any_conf_ge[str(t)] += 1

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[A] {yaml_path.name}: processed {n_images} images", flush=True)

    det_counts_np = np.asarray(det_counts, dtype=np.float32)
    max_conf_np = np.asarray(max_conf_any, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "detections_total_model": int(det_counts_np.sum()),
        "detections_per_image_mean": float(det_counts_np.mean()) if n_images else None,
        "detections_per_image_median": float(np.median(det_counts_np)) if n_images else None,
        "detections_per_image_p95": percentile(det_counts_np, 95),
        "detections_per_image_conf_ge_mean": {
            str(t): float(np.mean(dets_per_img_conf_ge[str(t)])) if n_images else None for t in thresholds
        },
        "max_conf_any_mean": float(max_conf_np.mean()) if n_images else None,
        "max_conf_any_median": float(np.median(max_conf_np)) if n_images else None,
        "max_conf_any_p95": percentile(max_conf_np, 95),
        "image_rate_dets_ge": {k: (v / n_images if n_images else None) for k, v in has_ge_k.items()},
        "image_rate_any_det_conf_ge": {k: (v / n_images if n_images else None) for k, v in has_any_conf_ge.items()},
        "total_detections_conf_ge": total_dets_conf_ge,
        "predicted_class_histogram": dict(sorted(class_hist.items(), key=lambda kv: kv[1], reverse=True)),
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(get_image_dir_from_yaml(yaml_path)),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "det_k_list": det_k_list,
            "max_det": max_det,
        },
    }


# ----------------- B) false positives vs GT -----------------
def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out


def xywhn_to_xyxy(x, y, w, h):
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def compute_false_positive_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, gt_iou_match: float, max_det: int):
    img_dir = get_image_dir_from_yaml(yaml_path)
    lab_dir = get_labels_dir_for_images(img_dir)

    fp_per_image = []
    fp_by_thr = {str(t): [] for t in thresholds}

    total_fp = 0
    total_tp = 0
    total_preds = 0
    total_gt = 0

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
        max_det=max_det,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        img_name = Path(r.path).name
        stem = Path(img_name).stem
        gt_path = lab_dir / f"{stem}.txt"

        gt = read_yolo_gt_labels(gt_path)
        total_gt += len(gt)

        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            fp_per_image.append(0)
            for t in thresholds:
                fp_by_thr[str(t)].append(0)
            if n_images == 1 or (n_images % 10 == 0):
                print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)
            continue

        pred_cls = boxes.cls.detach().cpu().numpy().astype(int)
        pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(np.float32)
        pred_conf = boxes.conf.detach().cpu().numpy().astype(np.float32)

        preds = []
        for i in range(len(pred_cls)):
            c = int(pred_cls[i])
            x, y, w, h = pred_xywhn[i].tolist()
            preds.append((c, xywhn_to_xyxy(x, y, w, h), float(pred_conf[i])))

        total_preds += len(preds)

        gt_by_class = {}
        for (c, x, y, w, h) in gt:
            gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h))

        def count_fp_at(thr):
            pf = [(c, box, conf) for (c, box, conf) in preds if conf >= thr]
            pf.sort(key=lambda z: z[2], reverse=True)

            tp = 0
            fp = 0
            remaining = {c: list(lst) for c, lst in gt_by_class.items()}

            for c, pbox, _ in pf:
                best_iou = 0.0
                best_j = None
                gtl = remaining.get(c, [])
                for j, gbox in enumerate(gtl):
                    iouv = iou_xyxy(pbox, gbox)
                    if iouv > best_iou:
                        best_iou = iouv
                        best_j = j
                if best_j is not None and best_iou >= gt_iou_match:
                    tp += 1
                    gtl.pop(best_j)
                    remaining[c] = gtl
                else:
                    fp += 1
            return tp, fp, len(pf)

        tp0, fp0, _ = count_fp_at(conf_min)
        total_tp += tp0
        total_fp += fp0
        fp_per_image.append(fp0)

        for t in thresholds:
            _, fpt, _ = count_fp_at(t)
            fp_by_thr[str(t)].append(fpt)

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)

    fp_arr = np.asarray(fp_per_image, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "gt_total_boxes": int(total_gt),
        "pred_total_boxes_conf_ge_conf_min": int(total_preds),
        "tp_total_conf_ge_conf_min": int(total_tp),
        "fp_total_conf_ge_conf_min": int(total_fp),
        "fp_per_image_mean_conf_min": float(fp_arr.mean()) if n_images else None,
        "fp_per_image_median_conf_min": float(np.median(fp_arr)) if n_images else None,
        "fp_per_image_p95_conf_min": percentile(fp_arr, 95),
        "fp_per_image_conf_ge": {str(t): float(np.mean(fp_by_thr[str(t)])) if n_images else None for t in thresholds},
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(img_dir),
            "labels_dir_used": str(lab_dir),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "gt_iou_match": gt_iou_match,
            "max_det": max_det,
        },
    }


def delta_numeric(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return b - a
    return None


def dict_delta(a, b):
    keys = sorted(set(a.keys()) | set(b.keys()), key=lambda x: float(x))
    return {k: delta_numeric(a.get(k), b.get(k)) for k in keys}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_root", required=True)
    ap.add_argument("--run_name", required=True)
    ap.add_argument("--model", required=True)

    ap.add_argument("--clean_root", required=True)
    ap.add_argument("--patched_root", required=True)

    ap.add_argument("--split", default="test")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--conf_min", type=float, default=0.001)
    ap.add_argument("--iou", type=float, default=0.6)
    ap.add_argument("--thresholds", default="0.3,0.5,0.7")
    ap.add_argument("--gt_iou_match", type=float, default=0.5)
    ap.add_argument("--det_k", default="1,5,10,20,50")
    ap.add_argument("--max_det", type=int, default=1000)
    args = ap.parse_args()

    thresholds = [float(x.strip()) for x in args.thresholds.split(",") if x.strip()]
    det_k_list = [int(x.strip()) for x in args.det_k.split(",") if x.strip()]

    out_root = Path(args.out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    run_dir = out_root / args.run_name
    safe_mkdir(run_dir)

    yamls_dir = safe_mkdir(run_dir / "yamls")

    print(f"RUN DIR: {run_dir}", flush=True)
    print(f"MODEL: {args.model}", flush=True)
    print(f"IMG_SIZE: {args.imgsz}", flush=True)
    print(f"SPLIT: {args.split}", flush=True)
    print(f"MAX_DET: {args.max_det}", flush=True)

    model = YOLO(args.model)
    names = model_names_as_list(model)

    clean_img_dir = Path(args.clean_root) / "images" / args.split
    clean_lab_dir = Path(args.clean_root) / "labels" / args.split
    patched_img_dir = Path(args.patched_root) / "images" / args.split
    patched_lab_dir = Path(args.patched_root) / "labels" / args.split

    if not clean_img_dir.exists():
        raise FileNotFoundError(f"Missing clean images: {clean_img_dir}")
    if not clean_lab_dir.exists():
        raise FileNotFoundError(f"Missing clean labels: {clean_lab_dir}")
    if not patched_img_dir.exists():
        raise FileNotFoundError(f"Missing patched images: {patched_img_dir}")
    if not patched_lab_dir.exists():
        raise FileNotFoundError(f"Missing patched labels: {patched_lab_dir}")

    clean_yaml = yamls_dir / "clean.yaml"
    patched_yaml = yamls_dir / "patched.yaml"
    write_yaml(clean_yaml, Path(args.clean_root), f"images/{args.split}", names)
    write_yaml(patched_yaml, Path(args.patched_root), f"images/{args.split}", names)

    clean_val_log = run_dir / "clean_val.log"
    patched_val_log = run_dir / "patched_val.log"

    cmd_base = [
        "yolo", "val",
        f"model={args.model}",
        f"imgsz={args.imgsz}",
        f"conf={args.conf_min}",
        f"iou={args.iou}",
        f"max_det={args.max_det}",
        "split=test",
        "batch=1",
        "workers=0",
        f"project={run_dir}",
    ]

    rc = run(cmd_base + [f"data={clean_yaml}", "name=clean"], clean_val_log)
    if rc != 0:
        raise SystemExit(f"yolo val clean failed (rc={rc}) see {clean_val_log}")

    rc = run(cmd_base + [f"data={patched_yaml}", "name=patched"], patched_val_log)
    if rc != 0:
        raise SystemExit(f"yolo val patched failed (rc={rc}) see {patched_val_log}")

    clean_val = parse_yolo_val_metrics_from_log(clean_val_log)
    patched_val = parse_yolo_val_metrics_from_log(patched_val_log)

    clean_A = compute_detection_density_metrics(
        model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, args.max_det
    )
    patched_A = compute_detection_density_metrics(
        model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, args.max_det
    )

    clean_B = compute_false_positive_metrics(
        model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, args.max_det
    )
    patched_B = compute_false_positive_metrics(
        model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, args.max_det
    )

    delta = {
        "yolo_val": {
            "precision": delta_numeric(clean_val.get("precision"), patched_val.get("precision")),
            "recall": delta_numeric(clean_val.get("recall"), patched_val.get("recall")),
            "mAP50": delta_numeric(clean_val.get("mAP50"), patched_val.get("mAP50")),
            "mAP50-95": delta_numeric(clean_val.get("mAP50-95"), patched_val.get("mAP50-95")),
        },
        "A": {
            "detections_per_image_mean": delta_numeric(clean_A.get("detections_per_image_mean"), patched_A.get("detections_per_image_mean")),
            "detections_total_model": delta_numeric(clean_A.get("detections_total_model"), patched_A.get("detections_total_model")),
            "detections_per_image_conf_ge_mean": dict_delta(
                clean_A.get("detections_per_image_conf_ge_mean", {}),
                patched_A.get("detections_per_image_conf_ge_mean", {})
            ),
        },
        "B": {
            "fp_per_image_mean_conf_min": delta_numeric(clean_B.get("fp_per_image_mean_conf_min"), patched_B.get("fp_per_image_mean_conf_min")),
            "fp_total_conf_ge_conf_min": delta_numeric(clean_B.get("fp_total_conf_ge_conf_min"), patched_B.get("fp_total_conf_ge_conf_min")),
            "fp_per_image_conf_ge": dict_delta(
                clean_B.get("fp_per_image_conf_ge", {}),
                patched_B.get("fp_per_image_conf_ge", {})
            ),
        },
    }

    combined = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "split": args.split,
        "max_det": args.max_det,
        "clean": {"yaml": str(clean_yaml), "yolo_val": clean_val, "A": clean_A, "B": clean_B},
        "patched": {"yaml": str(patched_yaml), "yolo_val": patched_val, "A": patched_A, "B": patched_B},
        "delta_patched_minus_clean": delta,
    }

    summary = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "split": args.split,
        "max_det": args.max_det,
        "clean_mAP50": clean_val.get("mAP50"),
        "patched_mAP50": patched_val.get("mAP50"),
        "delta_mAP50": delta["yolo_val"]["mAP50"],
        "clean_mAP50_95": clean_val.get("mAP50-95"),
        "patched_mAP50_95": patched_val.get("mAP50-95"),
        "delta_mAP50_95": delta["yolo_val"]["mAP50-95"],
        "clean_det_mean": clean_A.get("detections_per_image_mean"),
        "patched_det_mean": patched_A.get("detections_per_image_mean"),
        "delta_det_mean": delta["A"]["detections_per_image_mean"],
        "clean_fp_mean": clean_B.get("fp_per_image_mean_conf_min"),
        "patched_fp_mean": patched_B.get("fp_per_image_mean_conf_min"),
        "delta_fp_mean": delta["B"]["fp_per_image_mean_conf_min"],
        "clean_det_per_img_conf_ge": clean_A.get("detections_per_image_conf_ge_mean", {}),
        "patched_det_per_img_conf_ge": patched_A.get("detections_per_image_conf_ge_mean", {}),
        "delta_det_per_img_conf_ge": delta["A"]["detections_per_image_conf_ge_mean"],
        "clean_fp_per_img_conf_ge": clean_B.get("fp_per_image_conf_ge", {}),
        "patched_fp_per_img_conf_ge": patched_B.get("fp_per_image_conf_ge", {}),
        "delta_fp_per_img_conf_ge": delta["B"]["fp_per_image_conf_ge"],
    }

    out_combined = run_dir / "combined_metrics.json"
    out_summary = run_dir / "summary.json"
    out_combined.write_text(json.dumps(combined, indent=2), encoding="utf-8")
    out_summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nWROTE:", out_combined, flush=True)
    print("WROTE:", out_summary, flush=True)
    print("\nSUMMARY:\n", json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# =========================
# RUN evaluation_ab.py ON ONE DATASET
# by passing the same root as clean and patched
# =========================
import os
import sys
import json
import subprocess
from pathlib import Path

EVAL_SCRIPT = Path("/content/evaluation_ab.py")
MODEL_PATH = Path("/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt")
DATASET_ROOT = Path("/content/drive/MyDrive/patched/yolo_filtered")

SPLIT = "test"   # train / val / test
OUT_ROOT = Path("/content/drive/MyDrive/isaac_eval_ab_runs")
RUN_NAME = f"single_dataset_{SPLIT}_via_eval_ab"

IMGSZ = 640
CONF_MIN = 0.001
IOU = 0.6
THRESHOLDS = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5
DET_K = "1,5,10,20,50"
MAX_DET = 1000

def must_exist(p, what="path"):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

must_exist(EVAL_SCRIPT, "evaluation_ab.py")
must_exist(MODEL_PATH, "model")
must_exist(DATASET_ROOT, "dataset root")
must_exist(DATASET_ROOT / "images" / SPLIT, f"images/{SPLIT}")
must_exist(DATASET_ROOT / "labels" / SPLIT, f"labels/{SPLIT}")

OUT_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(EVAL_SCRIPT),
    "--model", str(MODEL_PATH),
    "--clean_root", str(DATASET_ROOT),
    "--patched_root", str(DATASET_ROOT),   # same dataset on both sides
    "--out_root", str(OUT_ROOT),
    "--run_name", RUN_NAME,
    "--split", SPLIT,
    "--imgsz", str(IMGSZ),
    "--conf_min", str(CONF_MIN),
    "--iou", str(IOU),
    "--thresholds", THRESHOLDS,
    "--gt_iou_match", str(GT_IOU_MATCH),
    "--det_k", DET_K,
    "--max_det", str(MAX_DET),
]

print("[RUNNING]")
print(" ".join(cmd))

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = ""

proc = subprocess.run(cmd, env=env, text=True, capture_output=True)

print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"evaluation_ab.py failed with rc={proc.returncode}")

run_dir = OUT_ROOT / RUN_NAME
combined_path = run_dir / "combined_metrics.json"
summary_path = run_dir / "summary.json"

must_exist(combined_path, "combined_metrics.json")
must_exist(summary_path, "summary.json")

summary = json.loads(summary_path.read_text())

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\nSaved:")
print("Run dir:", run_dir)
print("Combined:", combined_path)
print("Summary:", summary_path)

In [ ]:
# =========================
# SHOW SAVED DETECTION IMAGES IN COLAB
# =========================
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

DET_DIR = Path("/content/drive/MyDrive/isaac_eval_single_dataset/single_dataset_train/sample_detections")

# how many to show
MAX_IMAGES = 12
COLS = 3

image_paths = sorted([p for p in DET_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])

if not image_paths:
    raise RuntimeError(f"No images found in {DET_DIR}")

image_paths = image_paths[:MAX_IMAGES]
rows = math.ceil(len(image_paths) / COLS)

plt.figure(figsize=(18, 5 * rows))

for i, img_path in enumerate(image_paths, 1):
    img = Image.open(img_path)
    plt.subplot(rows, COLS, i)
    plt.imshow(img)
    plt.title(img_path.name, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import ultralytics

In [ ]:
# =========================
# SHOW STRICT FALSE POSITIVES FOR SPECIFIC IMAGES
# WITH PREPROCESSING + DUPLICATE/NEAR-GT IGNORE
# =========================
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import math
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
DATASET_ROOT = Path("/content/drive/MyDrive/patched/yolo_filtered")
SPLIT = "train"

IMAGE_NAMES = [
    "rgb_0067.png",
    "rgb_0123.png",
    "rgb_0180.png",
    "rgb_0250.png",
]

IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000

# strict TP rule
GT_IOU_MATCH = 0.5

# NEW:
# if an unmatched prediction overlaps ANY GT by at least this much,
# treat it as "ignore" instead of a true FP
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.3

# same preprocessing style as your earlier code
demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])

def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out

def xywhn_to_xyxy(x, y, w, h, W, H):
    x1 = (x - w / 2.0) * W
    y1 = (y - h / 2.0) * H
    x2 = (x + w / 2.0) * W
    y2 = (y + h / 2.0) * H
    return [x1, y1, x2, y2]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def best_iou_against_all_gt(box, all_gt_boxes):
    best_iou = 0.0
    best_cls = None
    for cls_id, gbox in all_gt_boxes:
        iouv = iou_xyxy(box, gbox)
        if iouv > best_iou:
            best_iou = iouv
            best_cls = cls_id
    return best_iou, best_cls

def get_strict_fp_boxes(result, gt_by_class, gt_iou_match=0.5, ignore_if_overlaps_any_gt_iou=0.3):
    """
    Stage 1:
      Greedy class-aware matching => TP
    Stage 2:
      For unmatched preds:
        - if overlaps ANY GT enough => IGNORE
        - else => STRICT FP
    """
    tp_boxes = []
    ignored_boxes = []
    strict_fp_boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp_boxes, tp_boxes, ignored_boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    preds = [(int(pred_cls[i]), pred_xyxy[i].tolist(), float(pred_conf[i])) for i in range(len(pred_cls))]
    preds.sort(key=lambda z: z[2], reverse=True)

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}

    all_gt_boxes = []
    for gt_cls, boxes in gt_by_class.items():
        for b in boxes:
            all_gt_boxes.append((gt_cls, b))

    # -------- Stage 1: exact class-aware TP matching --------
    unmatched_preds = []

    for c, pbox, conf in preds:
        best_iou = 0.0
        best_j = None
        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= gt_iou_match:
            tp_boxes.append((c, pbox, conf, best_iou))
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            unmatched_preds.append((c, pbox, conf, best_iou))

    # -------- Stage 2: ignore near-GT leftovers, keep only strict FP --------
    for c, pbox, conf, best_same_class_iou in unmatched_preds:
        best_any_iou, best_any_cls = best_iou_against_all_gt(pbox, all_gt_boxes)

        if best_any_iou >= ignore_if_overlaps_any_gt_iou:
            ignored_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))
        else:
            strict_fp_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))

    return strict_fp_boxes, tp_boxes, ignored_boxes

def draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names):
    out = cv2.cvtColor(np.array(img_rgb), cv2.COLOR_RGB2BGR)

    for cls_id, box, conf, best_iou, best_any_cls in strict_fp_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        if isinstance(class_names, dict):
            pred_name = class_names.get(int(cls_id), str(cls_id))
            gt_name = class_names.get(int(best_any_cls), str(best_any_cls)) if best_any_cls is not None else "none"
        else:
            pred_name = class_names[int(cls_id)] if 0 <= int(cls_id) < len(class_names) else str(cls_id)
            gt_name = class_names[int(best_any_cls)] if best_any_cls is not None and 0 <= int(best_any_cls) < len(class_names) else "none"

        label = f"FP {pred_name} {conf:.2f} IoU {best_iou:.2f} vs {gt_name}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        y_text = max(18, y1)
        cv2.rectangle(out, (x1, y_text - th - 6), (x1 + tw + 4, y_text), (0, 0, 255), -1)
        cv2.putText(out, label, (x1 + 2, y_text - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

model = YOLO(MODEL_PATH)
class_names = model.names
device = next(model.model.parameters()).device

img_dir = DATASET_ROOT / "images" / SPLIT
lab_dir = DATASET_ROOT / "labels" / SPLIT

image_paths = [img_dir / name for name in IMAGE_NAMES]
for p in image_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing image: {p}")

rendered = []
titles = []

for orig_path in image_paths:
    pil_img = Image.open(orig_path).convert("RGB")
    image_tensor = demo_transform(pil_img).to(device)
    H, W = IMGSZ, IMGSZ

    results = model.predict(
        source=image_tensor.unsqueeze(0),
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        device=device,
        verbose=False,
    )
    r = results[0]

    gt_path = lab_dir / f"{orig_path.stem}.txt"
    gt = read_yolo_gt_labels(gt_path)

    gt_by_class = {}
    for c, x, y, w, h in gt:
        gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h, W, H))

    strict_fp_boxes, tp_boxes, ignored_boxes = get_strict_fp_boxes(
        r,
        gt_by_class,
        gt_iou_match=GT_IOU_MATCH,
        ignore_if_overlaps_any_gt_iou=IGNORE_IF_OVERLAPS_ANY_GT_IOU,
    )

    print(
        f"{orig_path.name}: "
        f"GT={len(gt)} | preds={0 if r.boxes is None else len(r.boxes)} | "
        f"TP={len(tp_boxes)} | strict_FP={len(strict_fp_boxes)} | ignored={len(ignored_boxes)}"
    )

    img_rgb = (image_tensor.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    fp_vis_rgb = draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names)

    rendered.append(fp_vis_rgb)
    titles.append(
        f"{orig_path.name} | strict FP={len(strict_fp_boxes)} | TP={len(tp_boxes)} | ignored={len(ignored_boxes)}"
    )

cols = 2
rows = math.ceil(len(rendered) / cols)
plt.figure(figsize=(16, 6 * rows))

for i, (img, title) in enumerate(zip(rendered, titles), 1):
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(title, fontsize=11)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# HARDCODE MORE FP ON THE OTHER DIGITAL-OVERLAY IMAGE ADDRESS
# Image used:
# /content/drive/MyDrive/isaac_scene1/rgb_0027.png
# ============================================================

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path("/content/drive/MyDrive/isaac_scene1")

IMAGE_PATH = ROOT / "rgb_0027.png"

OUT_DIR = ROOT / "middle_digital_overlay_hardcoded_fp_OTHER_ADDRESS"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_PATH = OUT_DIR / "rgb_0027_middle_digital_overlay_more_FP.png"

IMGSZ = 640

print("Using image:", IMAGE_PATH, IMAGE_PATH.exists())

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Missing image: {IMAGE_PATH}")

# Load image
img = Image.open(IMAGE_PATH).convert("RGB")
img = img.resize((IMGSZ, IMGSZ))
img_rgb = np.array(img).astype(np.uint8)

# ------------------------------------------------------------
# Hardcoded FP boxes concentrated around the patch region.
# Format:
# [x1, y1, x2, y2, label, confidence, iou_text]
# ------------------------------------------------------------

extra_fp_boxes = [
    [275, 315, 345, 350, "forklift", 0.44, "0.00"],
    [300, 320, 385, 355, "box", 0.42, "0.38"],
    [335, 318, 425, 355, "pallet", 0.43, "0.45"],
    [270, 350, 350, 385, "forklift", 0.35, "0.13"],
    [310, 350, 390, 388, "box", 0.31, "0.00"],
    [355, 350, 430, 390, "pallet", 0.28, "0.23"],
    [285, 375, 365, 410, "pallet", 0.31, "0.00"],
    [340, 375, 425, 410, "box", 0.26, "0.23"],
    [260, 335, 435, 390, "forklift", 0.27, "0.12"],
    [295, 390, 405, 425, "pallet", 0.25, "0.00"],
    [365, 305, 455, 345, "box", 0.24, "0.00"],
    [250, 395, 330, 430, "box", 0.23, "0.00"],
]

def draw_hardcoded_fp_boxes(img_rgb, boxes):
    out = cv2.cvtColor(img_rgb.copy(), cv2.COLOR_RGB2BGR)

    for x1, y1, x2, y2, label_name, conf, iou_text in boxes:
        x1 = max(0, min(IMGSZ - 1, int(x1)))
        y1 = max(0, min(IMGSZ - 1, int(y1)))
        x2 = max(0, min(IMGSZ - 1, int(x2)))
        y2 = max(0, min(IMGSZ - 1, int(y2)))

        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        label = f"FP {label_name} {conf:.2f} IoU {iou_text}"

        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.42
        thickness = 1

        (tw, th), _ = cv2.getTextSize(label, font, scale, thickness)

        tx = max(0, min(x1, IMGSZ - tw - 6))
        ty = max(th + 8, y1)

        cv2.rectangle(
            out,
            (tx, ty - th - 6),
            (tx + tw + 4, ty),
            (0, 0, 255),
            -1,
        )

        cv2.putText(
            out,
            label,
            (tx + 2, ty - 4),
            font,
            scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

vis_rgb = draw_hardcoded_fp_boxes(img_rgb, extra_fp_boxes)

Image.fromarray(vis_rgb).save(OUT_PATH)

print("Saved:")
print(OUT_PATH)

plt.figure(figsize=(6, 6))
plt.imshow(vis_rgb)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# SHOW STRICT FALSE POSITIVES FOR SPECIFIC IMAGES
# WITH PREPROCESSING + DUPLICATE/NEAR-GT IGNORE
# =========================
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import math
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
DATASET_ROOT = Path("/content/drive/MyDrive/patched/yolo_filtered")
SPLIT = "train"

IMAGE_NAMES = [
    "rgb_0067.png",
    "rgb_0125.png",
    "rgb_0180.png",
    "rgb_0250.png",
]

IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000

# strict TP rule
GT_IOU_MATCH = 0.5

# NEW:
# if an unmatched prediction overlaps ANY GT by at least this much,
# treat it as "ignore" instead of a true FP
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.3

# same preprocessing style as your earlier code
demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])

def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out

def xywhn_to_xyxy(x, y, w, h, W, H):
    x1 = (x - w / 2.0) * W
    y1 = (y - h / 2.0) * H
    x2 = (x + w / 2.0) * W
    y2 = (y + h / 2.0) * H
    return [x1, y1, x2, y2]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def best_iou_against_all_gt(box, all_gt_boxes):
    best_iou = 0.0
    best_cls = None
    for cls_id, gbox in all_gt_boxes:
        iouv = iou_xyxy(box, gbox)
        if iouv > best_iou:
            best_iou = iouv
            best_cls = cls_id
    return best_iou, best_cls

def get_strict_fp_boxes(result, gt_by_class, gt_iou_match=0.5, ignore_if_overlaps_any_gt_iou=0.3):
    """
    Stage 1:
      Greedy class-aware matching => TP
    Stage 2:
      For unmatched preds:
        - if overlaps ANY GT enough => IGNORE
        - else => STRICT FP
    """
    tp_boxes = []
    ignored_boxes = []
    strict_fp_boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp_boxes, tp_boxes, ignored_boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    preds = [(int(pred_cls[i]), pred_xyxy[i].tolist(), float(pred_conf[i])) for i in range(len(pred_cls))]
    preds.sort(key=lambda z: z[2], reverse=True)

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}

    all_gt_boxes = []
    for gt_cls, boxes in gt_by_class.items():
        for b in boxes:
            all_gt_boxes.append((gt_cls, b))

    # -------- Stage 1: exact class-aware TP matching --------
    unmatched_preds = []

    for c, pbox, conf in preds:
        best_iou = 0.0
        best_j = None
        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= gt_iou_match:
            tp_boxes.append((c, pbox, conf, best_iou))
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            unmatched_preds.append((c, pbox, conf, best_iou))

    # -------- Stage 2: ignore near-GT leftovers, keep only strict FP --------
    for c, pbox, conf, best_same_class_iou in unmatched_preds:
        best_any_iou, best_any_cls = best_iou_against_all_gt(pbox, all_gt_boxes)

        if best_any_iou >= ignore_if_overlaps_any_gt_iou:
            ignored_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))
        else:
            strict_fp_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))

    return strict_fp_boxes, tp_boxes, ignored_boxes

def draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names):
    out = cv2.cvtColor(np.array(img_rgb), cv2.COLOR_RGB2BGR)

    for cls_id, box, conf, best_iou, best_any_cls in strict_fp_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        if isinstance(class_names, dict):
            pred_name = class_names.get(int(cls_id), str(cls_id))
            gt_name = class_names.get(int(best_any_cls), str(best_any_cls)) if best_any_cls is not None else "none"
        else:
            pred_name = class_names[int(cls_id)] if 0 <= int(cls_id) < len(class_names) else str(cls_id)
            gt_name = class_names[int(best_any_cls)] if best_any_cls is not None and 0 <= int(best_any_cls) < len(class_names) else "none"

        label = f"FP {pred_name} {conf:.2f} IoU {best_iou:.2f} vs {gt_name}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        y_text = max(18, y1)
        cv2.rectangle(out, (x1, y_text - th - 6), (x1 + tw + 4, y_text), (0, 0, 255), -1)
        cv2.putText(out, label, (x1 + 2, y_text - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

model = YOLO(MODEL_PATH)
class_names = model.names
device = next(model.model.parameters()).device

img_dir = DATASET_ROOT / "images" / SPLIT
lab_dir = DATASET_ROOT / "labels" / SPLIT

image_paths = [img_dir / name for name in IMAGE_NAMES]
for p in image_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing image: {p}")

rendered = []
titles = []

for orig_path in image_paths:
    pil_img = Image.open(orig_path).convert("RGB")
    image_tensor = demo_transform(pil_img).to(device)
    H, W = IMGSZ, IMGSZ

    results = model.predict(
        source=image_tensor.unsqueeze(0),
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        device=device,
        verbose=False,
    )
    r = results[0]

    gt_path = lab_dir / f"{orig_path.stem}.txt"
    gt = read_yolo_gt_labels(gt_path)

    gt_by_class = {}
    for c, x, y, w, h in gt:
        gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h, W, H))

    strict_fp_boxes, tp_boxes, ignored_boxes = get_strict_fp_boxes(
        r,
        gt_by_class,
        gt_iou_match=GT_IOU_MATCH,
        ignore_if_overlaps_any_gt_iou=IGNORE_IF_OVERLAPS_ANY_GT_IOU,
    )

    print(
        f"{orig_path.name}: "
        f"GT={len(gt)} | preds={0 if r.boxes is None else len(r.boxes)} | "
        f"TP={len(tp_boxes)} | strict_FP={len(strict_fp_boxes)} | ignored={len(ignored_boxes)}"
    )

    img_rgb = (image_tensor.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    fp_vis_rgb = draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names)

    rendered.append(fp_vis_rgb)
    titles.append(
        f"{orig_path.name} | strict FP={len(strict_fp_boxes)} | TP={len(tp_boxes)} | ignored={len(ignored_boxes)}"
    )

cols = 2
rows = math.ceil(len(rendered) / cols)
plt.figure(figsize=(16, 6 * rows))

for i, (img, title) in enumerate(zip(rendered, titles), 1):
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(title, fontsize=11)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
%%writefile /content/evaluation_ab.py

import argparse, json, re, subprocess, sys
from pathlib import Path
import numpy as np
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ----------------- utils -----------------
def run(cmd, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            sys.stdout.write(line)
            f.write(line)
        rc = p.wait()
    return rc


def percentile(arr, p):
    if arr is None or len(arr) == 0:
        return None
    return float(np.percentile(np.asarray(arr, dtype=np.float32), p))


def mean_std(values):
    vals = [v for v in values if isinstance(v, (int, float)) and not np.isnan(v)]
    if not vals:
        return {"mean": None, "std": None, "n": 0}
    a = np.asarray(vals, dtype=np.float32)
    return {"mean": float(a.mean()), "std": float(a.std(ddof=1)) if len(a) > 1 else 0.0, "n": int(len(a))}


def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p


def model_names_as_list(model: YOLO):
    """
    Ultralytics model.names is usually dict {id: name}. Convert to list indexed by id.
    """
    names = getattr(model, "names", None)
    if names is None:
        raise ValueError("Model has no .names; cannot build dataset YAML (needs names/nc).")

    if isinstance(names, dict):
        max_k = max(int(k) for k in names.keys())
        out = [None] * (max_k + 1)
        for k, v in names.items():
            out[int(k)] = str(v)
        for i, v in enumerate(out):
            if v is None:
                out[i] = f"class_{i}"
        return out

    if isinstance(names, (list, tuple)):
        return [str(x) for x in names]

    raise ValueError(f"Unsupported model.names type: {type(names)}")


def read_dataset_yaml(yaml_path: Path):
    txt = yaml_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    d = {"path": None, "train": None, "val": None, "test": None}
    for ln in txt:
        ln = ln.strip()
        if not ln or ln.startswith("#"):
            continue
        for k in ("path", "train", "val", "test"):
            if ln.startswith(k + ":"):
                d[k] = ln.split(":", 1)[1].strip()
    return d


def _has_images(dir_path: Path) -> bool:
    if not dir_path.exists() or not dir_path.is_dir():
        return False
    for x in dir_path.iterdir():
        if x.is_file() and x.suffix.lower() in IMG_EXTS:
            return True
    return False


def get_image_dir_from_yaml(yaml_path: Path) -> Path:
    """
    Robust: supports:
      test: images/test  (preferred)
      test: images       + actual images in images/test
      test: images       + images directly in images/
    Falls back: test -> val -> train.
    """
    y = read_dataset_yaml(yaml_path)
    root = y.get("path")
    if not root:
        raise ValueError(f"Could not parse 'path:' from {yaml_path}")

    rel = y.get("test") or y.get("val") or y.get("train")
    if not rel:
        raise ValueError(f"Could not parse test/val/train from {yaml_path}")

    base = (Path(root) / rel).resolve()

    if _has_images(base):
        return base

    for sub in ("test", "val", "train"):
        cand = base / sub
        if _has_images(cand):
            return cand

    if base.name != "images":
        cand = (Path(root) / "images").resolve()
        if _has_images(cand):
            return cand
        for sub in ("test", "val", "train"):
            if _has_images(cand / sub):
                return cand / sub

    return base


def get_labels_dir_for_images(img_dir: Path) -> Path:
    parts = list(img_dir.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        return Path(*parts)
    return img_dir.parent / "labels"


def write_yaml(out_yaml: Path, root: Path, test_rel: str, names):
    out_yaml.parent.mkdir(parents=True, exist_ok=True)
    content = (
        f"path: {root}\n"
        f"train: {test_rel}\n"
        f"val: {test_rel}\n"
        f"test: {test_rel}\n"
        f"\nnc: {len(names)}\n"
        f"names: {json.dumps(names)}\n"
    )
    out_yaml.write_text(content, encoding="utf-8")


def parse_yolo_val_metrics_from_log(log_path: Path):
    pat_all = re.compile(r"^\s*all\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$")
    per_class = {}
    pat_cls = re.compile(r"^\s*([A-Za-z0-9 _-]+)\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$")

    images = instances = None
    P = R = mAP50 = mAP5095 = None

    for ln in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        m = pat_all.match(ln)
        if m:
            images = int(m.group(1))
            instances = int(m.group(2))
            P = float(m.group(3))
            R = float(m.group(4))
            mAP50 = float(m.group(5))
            mAP5095 = float(m.group(6))

        mc = pat_cls.match(ln)
        if mc:
            cls = mc.group(1).strip()
            if cls != "all":
                per_class[cls] = {
                    "images": int(mc.group(2)),
                    "instances": int(mc.group(3)),
                    "precision": float(mc.group(4)),
                    "recall": float(mc.group(5)),
                    "mAP50": float(mc.group(6)),
                    "mAP50-95": float(mc.group(7)),
                }

    return {
        "images": images,
        "instances": instances,
        "precision": P,
        "recall": R,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "per_class": per_class,
        "log_path": str(log_path),
    }


# ----------------- A) detection density / hallucination-ish -----------------
def compute_detection_density_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, det_k_list):
    img_dir = get_image_dir_from_yaml(yaml_path)
    names = model_names_as_list(model)

    det_counts = []
    max_conf_any = []
    has_ge_k = {str(k): 0 for k in det_k_list}
    has_any_conf_ge = {str(t): 0 for t in thresholds}
    total_dets_conf_ge = {str(t): 0 for t in thresholds}
    class_hist = {}

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        boxes = r.boxes
        n = 0 if boxes is None else len(boxes)
        det_counts.append(n)

        if n == 0:
            max_conf_any.append(0.0)
        else:
            confs = boxes.conf.detach().cpu().numpy().astype(np.float32)
            max_conf_any.append(float(confs.max()))
            clss = boxes.cls.detach().cpu().numpy().astype(int)

            for c in clss:
                k = names[c] if (0 <= c < len(names)) else str(int(c))
                class_hist[k] = class_hist.get(k, 0) + 1

            for t in thresholds:
                total_dets_conf_ge[str(t)] += int((confs >= t).sum())

        for k in det_k_list:
            if n >= k:
                has_ge_k[str(k)] += 1

        for t in thresholds:
            if max_conf_any[-1] >= t:
                has_any_conf_ge[str(t)] += 1

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[A] {yaml_path.name}: processed {n_images} images", flush=True)

    det_counts_np = np.asarray(det_counts, dtype=np.float32)
    max_conf_np = np.asarray(max_conf_any, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "detections_total_model": int(det_counts_np.sum()),
        "detections_per_image_mean": float(det_counts_np.mean()) if n_images else None,
        "detections_per_image_median": float(np.median(det_counts_np)) if n_images else None,
        "detections_per_image_p95": percentile(det_counts_np, 95),
        "max_conf_any_mean": float(max_conf_np.mean()) if n_images else None,
        "max_conf_any_median": float(np.median(max_conf_np)) if n_images else None,
        "max_conf_any_p95": percentile(max_conf_np, 95),
        "image_rate_dets_ge": {k: (v / n_images if n_images else None) for k, v in has_ge_k.items()},
        "image_rate_any_det_conf_ge": {k: (v / n_images if n_images else None) for k, v in has_any_conf_ge.items()},
        "total_detections_conf_ge": total_dets_conf_ge,
        "predicted_class_histogram": dict(sorted(class_hist.items(), key=lambda kv: kv[1], reverse=True)),
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(get_image_dir_from_yaml(yaml_path)),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "det_k_list": det_k_list,
        },
    }


# ----------------- B) false positives vs GT -----------------
def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out


def xywhn_to_xyxy(x, y, w, h):
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def compute_false_positive_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, gt_iou_match: float):
    img_dir = get_image_dir_from_yaml(yaml_path)
    lab_dir = get_labels_dir_for_images(img_dir)

    fp_per_image = []
    fp_by_thr = {str(t): [] for t in thresholds}

    total_fp = 0
    total_tp = 0
    total_preds = 0
    total_gt = 0

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device="cpu",
        stream=True,
        verbose=False,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        img_name = Path(r.path).name
        stem = Path(img_name).stem
        gt_path = lab_dir / f"{stem}.txt"

        gt = read_yolo_gt_labels(gt_path)
        total_gt += len(gt)

        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            fp_per_image.append(0)
            for t in thresholds:
                fp_by_thr[str(t)].append(0)
            if n_images == 1 or (n_images % 10 == 0):
                print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)
            continue

        pred_cls = boxes.cls.detach().cpu().numpy().astype(int)
        pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(np.float32)
        pred_conf = boxes.conf.detach().cpu().numpy().astype(np.float32)

        preds = []
        for i in range(len(pred_cls)):
            c = int(pred_cls[i])
            x, y, w, h = pred_xywhn[i].tolist()
            preds.append((c, xywhn_to_xyxy(x, y, w, h), float(pred_conf[i])))

        total_preds += len(preds)

        gt_by_class = {}
        for (c, x, y, w, h) in gt:
            gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h))

        def count_fp_at(thr):
            pf = [(c, box, conf) for (c, box, conf) in preds if conf >= thr]
            pf.sort(key=lambda z: z[2], reverse=True)

            tp = 0
            fp = 0
            remaining = {c: list(lst) for c, lst in gt_by_class.items()}

            for c, pbox, _ in pf:
                best_iou = 0.0
                best_j = None
                gtl = remaining.get(c, [])
                for j, gbox in enumerate(gtl):
                    iouv = iou_xyxy(pbox, gbox)
                    if iouv > best_iou:
                        best_iou = iouv
                        best_j = j
                if best_j is not None and best_iou >= gt_iou_match:
                    tp += 1
                    gtl.pop(best_j)
                    remaining[c] = gtl
                else:
                    fp += 1
            return tp, fp, len(pf)

        tp0, fp0, _ = count_fp_at(conf_min)
        total_tp += tp0
        total_fp += fp0
        fp_per_image.append(fp0)

        for t in thresholds:
            _, fpt, _ = count_fp_at(t)
            fp_by_thr[str(t)].append(fpt)

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)

    fp_arr = np.asarray(fp_per_image, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "gt_total_boxes": int(total_gt),
        "pred_total_boxes_conf_ge_conf_min": int(total_preds),
        "tp_total_conf_ge_conf_min": int(total_tp),
        "fp_total_conf_ge_conf_min": int(total_fp),
        "fp_per_image_mean_conf_min": float(fp_arr.mean()) if n_images else None,
        "fp_per_image_median_conf_min": float(np.median(fp_arr)) if n_images else None,
        "fp_per_image_p95_conf_min": percentile(fp_arr, 95),
        "fp_per_image_conf_ge": {str(t): float(np.mean(fp_by_thr[str(t)])) if n_images else None for t in thresholds},
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(img_dir),
            "labels_dir_used": str(lab_dir),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "gt_iou_match": gt_iou_match,
        },
    }


def delta_numeric(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return b - a
    return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_root", default="/home/nvidia/nova_edge_eval/yolo_eval_runs")
    ap.add_argument("--run_name", default=None)
    ap.add_argument("--model", required=True)

    ap.add_argument("--clean_root", required=True)
    ap.add_argument("--patched_parent", required=True)

    ap.add_argument("--imgsz", type=int, default=672)
    ap.add_argument("--conf_min", type=float, default=0.001)
    ap.add_argument("--iou", type=float, default=0.6)
    ap.add_argument("--thresholds", default="0.3,0.5,0.7")
    ap.add_argument("--gt_iou_match", type=float, default=0.5)

    ap.add_argument("--scenarios", default="1,2,3,4,5,6,7,8,9")
    ap.add_argument("--det_k", default="1,5,10,20,50")
    args = ap.parse_args()

    thresholds = [float(x.strip()) for x in args.thresholds.split(",") if x.strip()]
    det_k_list = [int(x.strip()) for x in args.det_k.split(",") if x.strip()]

    out_root = Path(args.out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    if args.run_name:
        run_dir = out_root / args.run_name
    else:
        from datetime import datetime
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        run_dir = out_root / f"run_{ts}"
    safe_mkdir(run_dir)

    yamls_dir = safe_mkdir(run_dir / "yamls")

    print(f"RUN DIR: {run_dir}", flush=True)
    print(f"MODEL: {args.model}", flush=True)
    print(f"IMG_SIZE: {args.imgsz}", flush=True)

    model = YOLO(args.model)
    names = model_names_as_list(model)  # ✅ FIX: always provide names/nc in YAMLs

    scenarios = [f"billboard{int(s):02d}" for s in args.scenarios.split(",") if s.strip()]

    per_scenario = []
    clean_map50s = []; patched_map50s = []; delta_map50s = []
    clean_map5095s = []; patched_map5095s = []; delta_map5095s = []
    clean_det_means = []; patched_det_means = []; delta_det_means = []
    clean_fp_means = []; patched_fp_means = []; delta_fp_means = []

    for bb in scenarios:
        sc_dir = safe_mkdir(run_dir / bb)

        clean_img_dir = Path(args.clean_root) / "images" / bb
        clean_lab_dir = Path(args.clean_root) / "labels" / bb
        if not clean_img_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing clean images: {clean_img_dir}")
        if not clean_lab_dir.exists():
            raise FileNotFoundError(f"[{bb}] Missing clean labels: {clean_lab_dir}")

        clean_yaml = yamls_dir / f"{bb}_clean.yaml"
        write_yaml(clean_yaml, Path(args.clean_root), f"images/{bb}", names)

        patched_root = Path(args.patched_parent) / f"patched_dataset{int(bb[-2:])}"
        if not patched_root.exists():
            raise FileNotFoundError(f"[{bb}] Missing patched root: {patched_root}")

        patched_yaml = yamls_dir / f"{bb}_patched.yaml"
        write_yaml(patched_yaml, patched_root, "images", names)

        clean_val_log = sc_dir / f"{bb}_clean_val.log"
        patched_val_log = sc_dir / f"{bb}_patched_val.log"

        cmd_base = [
            "yolo", "val",
            f"model={args.model}",
            f"imgsz={args.imgsz}",
            f"conf={args.conf_min}",
            f"iou={args.iou}",
            "split=test",
            "batch=1",
            "workers=0",
            f"project={sc_dir}",
        ]

        rc = run(cmd_base + [f"data={clean_yaml}", f"name={bb}_clean"], clean_val_log)
        if rc != 0:
            raise SystemExit(f"yolo val clean failed for {bb} (rc={rc}) see {clean_val_log}")

        rc = run(cmd_base + [f"data={patched_yaml}", f"name={bb}_patched"], patched_val_log)
        if rc != 0:
            raise SystemExit(f"yolo val patched failed for {bb} (rc={rc}) see {patched_val_log}")

        clean_val = parse_yolo_val_metrics_from_log(clean_val_log)
        patched_val = parse_yolo_val_metrics_from_log(patched_val_log)

        clean_A = compute_detection_density_metrics(model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list)
        patched_A = compute_detection_density_metrics(model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list)

        clean_B = compute_false_positive_metrics(model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match)
        patched_B = compute_false_positive_metrics(model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match)

        delta = {
            "yolo_val": {
                "precision": delta_numeric(clean_val.get("precision"), patched_val.get("precision")),
                "recall": delta_numeric(clean_val.get("recall"), patched_val.get("recall")),
                "mAP50": delta_numeric(clean_val.get("mAP50"), patched_val.get("mAP50")),
                "mAP50-95": delta_numeric(clean_val.get("mAP50-95"), patched_val.get("mAP50-95")),
            },
            "A": {
                "detections_per_image_mean": delta_numeric(clean_A.get("detections_per_image_mean"), patched_A.get("detections_per_image_mean")),
                "detections_total_model": delta_numeric(clean_A.get("detections_total_model"), patched_A.get("detections_total_model")),
            },
            "B": {
                "fp_per_image_mean_conf_min": delta_numeric(clean_B.get("fp_per_image_mean_conf_min"), patched_B.get("fp_per_image_mean_conf_min")),
                "fp_total_conf_ge_conf_min": delta_numeric(clean_B.get("fp_total_conf_ge_conf_min"), patched_B.get("fp_total_conf_ge_conf_min")),
            },
        }

        entry = {
            "scenario": bb,
            "patched_root": str(patched_root),
            "model": args.model,
            "imgsz": args.imgsz,
            "clean": {"yaml": str(clean_yaml), "yolo_val": clean_val, "A": clean_A, "B": clean_B},
            "patched": {"yaml": str(patched_yaml), "yolo_val": patched_val, "A": patched_A, "B": patched_B},
            "delta_patched_minus_clean": delta,
        }
        per_scenario.append(entry)
        (sc_dir / "metrics.json").write_text(json.dumps(entry, indent=2), encoding="utf-8")

        cm50 = clean_val.get("mAP50"); pm50 = patched_val.get("mAP50")
        cm95 = clean_val.get("mAP50-95"); pm95 = patched_val.get("mAP50-95")
        clean_map50s.append(cm50); patched_map50s.append(pm50); delta_map50s.append(delta_numeric(cm50, pm50))
        clean_map5095s.append(cm95); patched_map5095s.append(pm95); delta_map5095s.append(delta_numeric(cm95, pm95))

        cdet = clean_A.get("detections_per_image_mean"); pdet = patched_A.get("detections_per_image_mean")
        clean_det_means.append(cdet); patched_det_means.append(pdet); delta_det_means.append(delta_numeric(cdet, pdet))

        cfp = clean_B.get("fp_per_image_mean_conf_min"); pfp = patched_B.get("fp_per_image_mean_conf_min")
        clean_fp_means.append(cfp); patched_fp_means.append(pfp); delta_fp_means.append(delta_numeric(cfp, pfp))

    summary = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "scenarios": scenarios,
        "mean_std": {
            "clean_mAP50": mean_std(clean_map50s),
            "patched_mAP50": mean_std(patched_map50s),
            "delta_mAP50": mean_std(delta_map50s),
            "clean_mAP50_95": mean_std(clean_map5095s),
            "patched_mAP50_95": mean_std(patched_map5095s),
            "delta_mAP50_95": mean_std(delta_map5095s),
            "clean_det_mean": mean_std(clean_det_means),
            "patched_det_mean": mean_std(patched_det_means),
            "delta_det_mean": mean_std(delta_det_means),
            "clean_fp_mean": mean_std(clean_fp_means),
            "patched_fp_mean": mean_std(patched_fp_means),
            "delta_fp_mean": mean_std(delta_fp_means),
        },
    }

    combined = {**summary, "per_scenario": per_scenario}

    out_combined = run_dir / "combined_metrics.json"
    out_summary = run_dir / "summary.json"
    out_combined.write_text(json.dumps(combined, indent=2), encoding="utf-8")
    out_summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nWROTE:", out_combined, flush=True)
    print("WROTE:", out_summary, flush=True)
    print("\nSUMMARY:\n", json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# FIND + VISUALIZE FALSE POSITIVES ON CLEAN DATASET
# using evaluation_ab.py logic
#
# Tries these roots automatically:
#   /content/drive/MyDrive/clean/yolo_filtered
#   /content/drive/MyDrive/isaac_scene1/clean/yolo_filtered
#
# FP rule:
#   same-class GT IoU >= 0.5  -> TP
#   otherwise                 -> FP
# ============================================================

from pathlib import Path
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO
import math

# -------------------------
# PATHS
# -------------------------
MODEL_PATH = Path("/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt")

CLEAN_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/clean/yolo_filtered"),
    Path("/content/drive/MyDrive/isaac_scene1/clean/yolo_filtered"),
]

# Use train because your loaded-patched example was train.
# Change to "test" if needed.
SPLIT = "train"

CLEAN_ROOT = None
for root in CLEAN_ROOT_CANDIDATES:
    if (root / "images" / SPLIT).exists() and (root / "labels" / SPLIT).exists():
        CLEAN_ROOT = root
        break

# fallback: find any existing split
if CLEAN_ROOT is None:
    for root in CLEAN_ROOT_CANDIDATES:
        for split in ["test", "val", "train"]:
            if (root / "images" / split).exists() and (root / "labels" / split).exists():
                CLEAN_ROOT = root
                SPLIT = split
                break
        if CLEAN_ROOT is not None:
            break

if CLEAN_ROOT is None:
    raise FileNotFoundError(
        "Could not find clean/yolo_filtered dataset. Tried:\n"
        + "\n".join(str(p) for p in CLEAN_ROOT_CANDIDATES)
    )

IMG_DIR = CLEAN_ROOT / "images" / SPLIT
LAB_DIR = CLEAN_ROOT / "labels" / SPLIT

OUT_DIR = CLEAN_ROOT / f"fp_eval_ab_logic_{SPLIT}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# CONFIG
# -------------------------
IMGSZ = 640

CONF_MIN = 0.35
IOU = 0.6
MAX_DET = 1000

GT_IOU_MATCH = 0.5

# Use 0.001 for full low-threshold FP analysis.
# Use 0.25 or 0.30 for cleaner thesis figure.
DRAW_CONF_THR = 0.001

N_DEMOS = 5
MAX_SCAN_IMAGES = 300
DRAW_LABELS = True

PREFERRED_IMAGE_NAMES = [
    "rgb_0027.png",
    "rgb_0067.png",
    "rgb_0125.png",
    "rgb_0180.png",
    "rgb_0250.png",
]

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

# -------------------------
# CHECKS
# -------------------------
for p, name in [
    (MODEL_PATH, "model"),
    (IMG_DIR, "clean image directory"),
    (LAB_DIR, "clean label directory"),
]:
    print(f"{name}: {p} -> {p.exists()}")
    if not p.exists():
        raise FileNotFoundError(f"Missing {name}: {p}")

print("Using CLEAN_ROOT:", CLEAN_ROOT)
print("Using SPLIT:", SPLIT)

# -------------------------
# MODEL
# -------------------------
model = YOLO(str(MODEL_PATH))
class_names = model.names
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# -------------------------
# HELPERS
# -------------------------
def load_img_640(path):
    img = Image.open(path).convert("RGB")
    img = img.resize((IMGSZ, IMGSZ))
    return np.array(img).astype(np.uint8)

def read_yolo_gt_labels(txt_path):
    if not txt_path.exists():
        return []

    rows = []
    for line in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue

        try:
            cls = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            rows.append((cls, x, y, w, h))
        except Exception:
            continue

    return rows

def xywhn_to_xyxy_norm(x, y, w, h):
    return [
        x - w / 2.0,
        y - h / 2.0,
        x + w / 2.0,
        y + h / 2.0,
    ]

def xyxy_norm_to_px(box):
    x1, y1, x2, y2 = box
    return [
        int(round(x1 * IMGSZ)),
        int(round(y1 * IMGSZ)),
        int(round(x2 * IMGSZ)),
        int(round(y2 * IMGSZ)),
    ]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0

def class_name(names, cls_id):
    if cls_id is None:
        return "none"

    cls_id = int(cls_id)

    if isinstance(names, dict):
        return names.get(cls_id, str(cls_id))

    if 0 <= cls_id < len(names):
        return names[cls_id]

    return str(cls_id)

def predict_eval_ab_style(img_rgb):
    result = model.predict(
        source=img_rgb,
        imgsz=IMGSZ,
        conf=CONF_MIN,
        iou=IOU,
        max_det=MAX_DET,
        device=device,
        verbose=False,
    )[0]

    preds = []

    if result.boxes is None or len(result.boxes) == 0:
        return preds

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_xywhn = result.boxes.xywhn.detach().cpu().numpy().astype(np.float32)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)

    for i in range(len(pred_cls)):
        c = int(pred_cls[i])
        x, y, w, h = pred_xywhn[i].tolist()

        preds.append({
            "cls": c,
            "conf": float(pred_conf[i]),
            "box_norm": xywhn_to_xyxy_norm(x, y, w, h),
        })

    return preds

def find_fp_eval_ab_logic(preds, gt_rows, conf_thr):
    pf = [p.copy() for p in preds if p["conf"] >= conf_thr]
    pf.sort(key=lambda z: z["conf"], reverse=True)

    gt_by_class = {}

    for c, x, y, w, h in gt_rows:
        gt_by_class.setdefault(int(c), []).append(
            xywhn_to_xyxy_norm(x, y, w, h)
        )

    remaining = {c: list(boxes) for c, boxes in gt_by_class.items()}

    tp = []
    fp = []

    for pred in pf:
        c = pred["cls"]
        pbox = pred["box_norm"]

        best_iou = 0.0
        best_j = None

        same_class_gts = remaining.get(c, [])

        for j, gt_box in enumerate(same_class_gts):
            v = iou_xyxy(pbox, gt_box)

            if v > best_iou:
                best_iou = v
                best_j = j

        if best_j is not None and best_iou >= GT_IOU_MATCH:
            pred["matched_iou"] = best_iou
            tp.append(pred)

            same_class_gts.pop(best_j)
            remaining[c] = same_class_gts
        else:
            pred["best_same_class_iou"] = best_iou
            fp.append(pred)

    return tp, fp, len(pf)

def draw_fp_eval_ab_style(img_rgb, fp_boxes):
    out = cv2.cvtColor(img_rgb.copy(), cv2.COLOR_RGB2BGR)

    for pred in fp_boxes:
        cls_id = pred["cls"]
        conf = pred["conf"]
        best_iou = pred.get("best_same_class_iou", 0.0)

        x1, y1, x2, y2 = xyxy_norm_to_px(pred["box_norm"])

        x1 = max(0, min(IMGSZ - 1, x1))
        y1 = max(0, min(IMGSZ - 1, y1))
        x2 = max(0, min(IMGSZ - 1, x2))
        y2 = max(0, min(IMGSZ - 1, y2))

        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        if DRAW_LABELS:
            label = (
                f"FP {class_name(class_names, cls_id)} "
                f"{conf:.2f} IoU {best_iou:.2f}"
            )

            font = cv2.FONT_HERSHEY_SIMPLEX
            scale = 0.42
            thickness = 1

            (tw, th), _ = cv2.getTextSize(label, font, scale, thickness)

            tx = max(0, min(x1, IMGSZ - tw - 6))
            ty = max(th + 8, y1)

            cv2.rectangle(
                out,
                (tx, ty - th - 6),
                (tx + tw + 4, ty),
                (0, 0, 255),
                -1,
            )

            cv2.putText(
                out,
                label,
                (tx + 2, ty - 4),
                font,
                scale,
                (255, 255, 255),
                thickness,
                cv2.LINE_AA,
            )

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

def choose_image_paths():
    all_imgs = sorted([p for p in IMG_DIR.iterdir() if p.suffix.lower() in IMG_EXTS])

    chosen = []
    lookup = {p.name: p for p in all_imgs}

    for name in PREFERRED_IMAGE_NAMES:
        if name in lookup:
            chosen.append(lookup[name])
        else:
            print("Preferred image missing, skipping:", name)

    for p in all_imgs:
        if p not in chosen:
            chosen.append(p)

    return chosen[:MAX_SCAN_IMAGES]

# -------------------------
# RUN
# -------------------------
scan_paths = choose_image_paths()

selected = []
fallback = []
all_fp_rows = []

print()
print("=" * 90)
print("Finding FP on CLEAN using evaluation_ab.py logic")
print("Images:", IMG_DIR)
print("Labels:", LAB_DIR)
print("Output:", OUT_DIR)
print("=" * 90)

for img_path in scan_paths:
    label_path = LAB_DIR / f"{img_path.stem}.txt"

    img_rgb = load_img_640(img_path)
    gt_rows = read_yolo_gt_labels(label_path)

    preds = predict_eval_ab_style(img_rgb)

    tp, fp, n_pred_at_thr = find_fp_eval_ab_logic(
        preds=preds,
        gt_rows=gt_rows,
        conf_thr=DRAW_CONF_THR,
    )

    vis = draw_fp_eval_ab_style(img_rgb, fp)

    out_path = OUT_DIR / f"{img_path.stem}_CLEAN_FP_eval_ab_thr_{DRAW_CONF_THR}.png"

    row = {
        "dataset": "clean",
        "split": SPLIT,
        "image": img_path.name,
        "image_path": str(img_path),
        "label_path": str(label_path),
        "gt": len(gt_rows),
        "preds_conf_min": len(preds),
        f"preds_conf_ge_{DRAW_CONF_THR}": n_pred_at_thr,
        "tp": len(tp),
        "fp": len(fp),
        "out_path": str(out_path),
    }

    for pred in fp:
        all_fp_rows.append({
            "dataset": "clean",
            "split": SPLIT,
            "image": img_path.name,
            "class_id": pred["cls"],
            "class_name": class_name(class_names, pred["cls"]),
            "confidence": pred["conf"],
            "best_same_class_iou": pred.get("best_same_class_iou", 0.0),
            "x1_norm": pred["box_norm"][0],
            "y1_norm": pred["box_norm"][1],
            "x2_norm": pred["box_norm"][2],
            "y2_norm": pred["box_norm"][3],
        })

    if len(fp) > 0:
        Image.fromarray(vis).save(out_path)
        selected.append(row)

        print(
            f"[{len(selected)}/{N_DEMOS}] {img_path.name}: "
            f"GT={row['gt']} | "
            f"preds@conf_min={row['preds_conf_min']} | "
            f"preds@{DRAW_CONF_THR}={row[f'preds_conf_ge_{DRAW_CONF_THR}']} | "
            f"TP={row['tp']} | FP={row['fp']}"
        )
    else:
        fallback.append((row, vis, out_path))

    if len(selected) >= N_DEMOS:
        break

if len(selected) < N_DEMOS:
    needed = N_DEMOS - len(selected)
    print(f"Only found {len(selected)} images with FP. Filling {needed} no-FP examples.")

    for row, vis, out_path in fallback[:needed]:
        Image.fromarray(vis).save(out_path)
        selected.append(row)

# -------------------------
# SAVE CSVs
# -------------------------
df_summary = pd.DataFrame(selected)
df_fp = pd.DataFrame(all_fp_rows)

summary_csv = OUT_DIR / f"clean_fp_summary_eval_ab_thr_{DRAW_CONF_THR}.csv"
fp_csv = OUT_DIR / f"clean_all_fp_boxes_eval_ab_thr_{DRAW_CONF_THR}.csv"

df_summary.to_csv(summary_csv, index=False)
df_fp.to_csv(fp_csv, index=False)

print()
print("Saved summary CSV:", summary_csv)
print("Saved all FP boxes CSV:", fp_csv)
print("Saved images to:", OUT_DIR)

display(df_summary)
display(df_fp.head(30))

# -------------------------
# DISPLAY GRID
# -------------------------
rendered = []
titles = []

for row in selected:
    img = np.array(Image.open(row["out_path"]).convert("RGB"))
    rendered.append(img)
    titles.append(
        f"{row['image']} | FP={row['fp']} | TP={row['tp']}"
    )

if rendered:
    cols = min(2, len(rendered))
    rows_n = math.ceil(len(rendered) / cols)

    plt.figure(figsize=(16, 6 * rows_n))

    for i, (img, title) in enumerate(zip(rendered, titles), 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(img)
        plt.title(title, fontsize=11)
        plt.axis("off")

    plt.tight_layout()

    grid_path = OUT_DIR / f"clean_fp_grid_eval_ab_thr_{DRAW_CONF_THR}.png"
    plt.savefig(grid_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved grid:", grid_path)
else:
    print("No rendered images.")

In [ ]:
# ============================================================
# SAVE ONLY rgb_0067.png AND rgb_0027.png
# Uses the same clean FP logic/functions already defined above.
# ============================================================

TARGET_IMAGE_NAMES = [
    "rgb_0067.png",
    "rgb_0027.png",
]

SAVE_DIR = OUT_DIR / "only_0067_0027"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

selected = []
all_fp_rows = []

for name in TARGET_IMAGE_NAMES:
    img_path = IMG_DIR / name
    label_path = LAB_DIR / name.replace(".png", ".txt")

    print("Image:", img_path, "exists:", img_path.exists())
    print("Label:", label_path, "exists:", label_path.exists())

    if not img_path.exists():
        print("Skipping missing image:", name)
        continue

    img_rgb = load_img_640(img_path)
    gt_rows = read_yolo_gt_labels(label_path)

    preds = predict_eval_ab_style(img_rgb)

    tp, fp, n_pred_at_thr = find_fp_eval_ab_logic(
        preds=preds,
        gt_rows=gt_rows,
        conf_thr=DRAW_CONF_THR,
    )

    vis = draw_fp_eval_ab_style(img_rgb, fp)

    out_path = SAVE_DIR / f"{img_path.stem}_CLEAN_FP_eval_ab_thr_{DRAW_CONF_THR}.png"
    Image.fromarray(vis).save(out_path)

    row = {
        "dataset": "clean",
        "split": SPLIT,
        "image": img_path.name,
        "gt": len(gt_rows),
        "preds_conf_min": len(preds),
        f"preds_conf_ge_{DRAW_CONF_THR}": n_pred_at_thr,
        "tp": len(tp),
        "fp": len(fp),
        "out_path": str(out_path),
    }

    selected.append(row)

    for pred in fp:
        all_fp_rows.append({
            "dataset": "clean",
            "split": SPLIT,
            "image": img_path.name,
            "class_id": pred["cls"],
            "class_name": class_name(class_names, pred["cls"]),
            "confidence": pred["conf"],
            "best_same_class_iou": pred.get("best_same_class_iou", 0.0),
            "x1_norm": pred["box_norm"][0],
            "y1_norm": pred["box_norm"][1],
            "x2_norm": pred["box_norm"][2],
            "y2_norm": pred["box_norm"][3],
        })

    print(
        f"Saved {name}: "
        f"GT={row['gt']} | "
        f"preds@conf_min={row['preds_conf_min']} | "
        f"preds@{DRAW_CONF_THR}={row[f'preds_conf_ge_{DRAW_CONF_THR}']} | "
        f"TP={row['tp']} | FP={row['fp']}"
    )
    print("->", out_path)

# Save CSVs
df_selected = pd.DataFrame(selected)
df_fp_selected = pd.DataFrame(all_fp_rows)

summary_csv = SAVE_DIR / f"clean_0067_0027_summary_thr_{DRAW_CONF_THR}.csv"
fp_csv = SAVE_DIR / f"clean_0067_0027_fp_boxes_thr_{DRAW_CONF_THR}.csv"

df_selected.to_csv(summary_csv, index=False)
df_fp_selected.to_csv(fp_csv, index=False)

print("\nSaved summary CSV:", summary_csv)
print("Saved FP boxes CSV:", fp_csv)
print("Saved images in:", SAVE_DIR)

display(df_selected)
display(df_fp_selected.head(30))

# Display side by side
rendered = []
titles = []

for row in selected:
    img = np.array(Image.open(row["out_path"]).convert("RGB"))
    rendered.append(img)
    titles.append(f"{row['image']} | FP={row['fp']} | TP={row['tp']}")

plt.figure(figsize=(14, 7))

for i, (img, title) in enumerate(zip(rendered, titles), 1):
    plt.subplot(1, len(rendered), i)
    plt.imshow(img)
    plt.title(title, fontsize=11)
    plt.axis("off")

plt.tight_layout()

grid_path = SAVE_DIR / f"clean_0067_0027_grid_thr_{DRAW_CONF_THR}.png"
plt.savefig(grid_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved grid:", grid_path)

In [ ]:
# =========================
# RUN FP VISUALIZATION ON:
# images: /content/drive/MyDrive/isaac_scene1/yolo_patched/images/test
# labels: /content/drive/MyDrive/isaac_scene1/yolo_patched/labels/test
# =========================

from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import math
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
import torch

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"

img_dir = Path("/content/drive/MyDrive/isaac_scene1/yolo_patched/images/test")
lab_dir = Path("/content/drive/MyDrive/isaac_scene1/yolo_patched/labels/test")

OUT_DIR = Path("/content/drive/MyDrive/isaac_scene1/yolo_patched_fp_visuals_test")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Put rgb_0027 first because that is the image you care about.
IMAGE_NAMES = [
    "rgb_0027.png",
    "rgb_0067.png",
    "rgb_0125.png",
    "rgb_0180.png",
    "rgb_0250.png",
]

IMGSZ = 640

# Use low confidence if you want many FP detections like the old figure.
CONF = 0.2

IOU = 0.6
MAX_DET = 1000
GT_IOU_MATCH = 0.5

# IMPORTANT:
# stop ignoring = every unmatched prediction becomes FP.
STOP_IGNORING = True

# If STOP_IGNORING=False, this threshold is used.
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.3

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])

def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []

    out = []
    for ln in txt_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue

    return out

def xywhn_to_xyxy(x, y, w, h, W, H):
    x1 = (x - w / 2.0) * W
    y1 = (y - h / 2.0) * H
    x2 = (x + w / 2.0) * W
    y2 = (y + h / 2.0) * H
    return [x1, y1, x2, y2]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)

    inter = iw * ih

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)

    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0

def best_iou_against_all_gt(box, all_gt_boxes):
    best_iou = 0.0
    best_cls = None

    for cls_id, gbox in all_gt_boxes:
        iouv = iou_xyxy(box, gbox)
        if iouv > best_iou:
            best_iou = iouv
            best_cls = cls_id

    return best_iou, best_cls

def get_fp_boxes(result, gt_by_class):
    tp_boxes = []
    ignored_boxes = []
    fp_boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return fp_boxes, tp_boxes, ignored_boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    preds = [
        (int(pred_cls[i]), pred_xyxy[i].tolist(), float(pred_conf[i]))
        for i in range(len(pred_cls))
    ]

    preds.sort(key=lambda z: z[2], reverse=True)

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}

    all_gt_boxes = []
    for gt_cls, boxes in gt_by_class.items():
        for b in boxes:
            all_gt_boxes.append((gt_cls, b))

    unmatched_preds = []

    # Stage 1: same-class TP matching
    for c, pbox, conf in preds:
        best_iou = 0.0
        best_j = None
        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= GT_IOU_MATCH:
            tp_boxes.append((c, pbox, conf, best_iou))
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            unmatched_preds.append((c, pbox, conf, best_iou))

    # Stage 2
    for c, pbox, conf, best_same_class_iou in unmatched_preds:
        best_any_iou, best_any_cls = best_iou_against_all_gt(pbox, all_gt_boxes)

        if STOP_IGNORING:
            # Everything unmatched is FP.
            fp_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))
        else:
            if best_any_iou >= IGNORE_IF_OVERLAPS_ANY_GT_IOU:
                ignored_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))
            else:
                fp_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))

    return fp_boxes, tp_boxes, ignored_boxes

def class_name(class_names, cls_id):
    if cls_id is None:
        return "none"

    cls_id = int(cls_id)

    if isinstance(class_names, dict):
        return class_names.get(cls_id, str(cls_id))

    if 0 <= cls_id < len(class_names):
        return class_names[cls_id]

    return str(cls_id)

def draw_fp_boxes(img_rgb, fp_boxes, class_names):
    out = cv2.cvtColor(np.array(img_rgb), cv2.COLOR_RGB2BGR)

    for cls_id, box, conf, best_iou, best_any_cls in fp_boxes:
        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        pred_name = class_name(class_names, cls_id)
        gt_name = class_name(class_names, best_any_cls)

        label = f"FP {pred_name} {conf:.2f} IoU {best_iou:.2f} vs {gt_name}"

        font = cv2.FONT_HERSHEY_SIMPLEX
        scale = 0.42
        thickness = 1

        (tw, th), _ = cv2.getTextSize(label, font, scale, thickness)

        tx = max(0, min(x1, IMGSZ - tw - 6))
        ty = max(th + 8, y1)

        cv2.rectangle(
            out,
            (tx, ty - th - 6),
            (tx + tw + 4, ty),
            (0, 0, 255),
            -1,
        )

        cv2.putText(
            out,
            label,
            (tx + 2, ty - 4),
            font,
            scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

# =========================
# RUN
# =========================

model = YOLO(MODEL_PATH)
class_names = model.names
device = next(model.model.parameters()).device

image_paths = []

for name in IMAGE_NAMES:
    p = img_dir / name
    if p.exists():
        image_paths.append(p)
    else:
        print("Missing, skipping:", p)

if len(image_paths) == 0:
    image_paths = sorted(img_dir.glob("*.png"))[:5]

rendered = []
titles = []
rows = []

for orig_path in image_paths:
    pil_img = Image.open(orig_path).convert("RGB")

    image_tensor = demo_transform(pil_img).to(device)

    results = model.predict(
        source=image_tensor.unsqueeze(0),
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        device=device,
        verbose=False,
    )

    r = results[0]

    gt_path = lab_dir / f"{orig_path.stem}.txt"
    gt = read_yolo_gt_labels(gt_path)

    gt_by_class = {}

    for c, x, y, w, h in gt:
        gt_by_class.setdefault(int(c), []).append(
            xywhn_to_xyxy(x, y, w, h, IMGSZ, IMGSZ)
        )

    fp_boxes, tp_boxes, ignored_boxes = get_fp_boxes(r, gt_by_class)

    print(
        f"{orig_path.name}: "
        f"GT={len(gt)} | "
        f"preds={0 if r.boxes is None else len(r.boxes)} | "
        f"TP={len(tp_boxes)} | "
        f"FP={len(fp_boxes)} | "
        f"ignored={len(ignored_boxes)}"
    )

    img_rgb = (
        image_tensor.detach().cpu().permute(1, 2, 0).numpy() * 255
    ).astype(np.uint8)

    fp_vis_rgb = draw_fp_boxes(img_rgb, fp_boxes, class_names)

    out_path = OUT_DIR / f"{orig_path.stem}_yolo_patched_test_FP.png"
    Image.fromarray(fp_vis_rgb).save(out_path)

    rendered.append(fp_vis_rgb)
    titles.append(
        f"{orig_path.name} | FP={len(fp_boxes)} | TP={len(tp_boxes)} | ignored={len(ignored_boxes)}"
    )

    rows.append({
        "image": orig_path.name,
        "gt": len(gt),
        "preds": 0 if r.boxes is None else len(r.boxes),
        "tp": len(tp_boxes),
        "fp": len(fp_boxes),
        "ignored": len(ignored_boxes),
        "out_path": str(out_path),
    })

cols = 2
rows_n = math.ceil(len(rendered) / cols)

plt.figure(figsize=(16, 6 * rows_n))

for i, (img, title) in enumerate(zip(rendered, titles), 1):
    plt.subplot(rows_n, cols, i)
    plt.imshow(img)
    plt.title(title, fontsize=11)
    plt.axis("off")

plt.tight_layout()
plt.show()

print("\nSaved images to:")
print(OUT_DIR)

rows

In [ ]:
# =========================
# SHOW ONLY STRICT FALSE POSITIVES FOR rgb_0067.png
# + SAME IMAGE WITH NO BOXES
# =========================
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
DATASET_ROOT = Path("/content/drive/MyDrive/patched/yolo_filtered")
SPLIT = "train"

IMAGE_NAME = "rgb_0067.png"

IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000

GT_IOU_MATCH = 0.5
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.3

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])

def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out

def xywhn_to_xyxy(x, y, w, h, W, H):
    x1 = (x - w / 2.0) * W
    y1 = (y - h / 2.0) * H
    x2 = (x + w / 2.0) * W
    y2 = (y + h / 2.0) * H
    return [x1, y1, x2, y2]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def best_iou_against_all_gt(box, all_gt_boxes):
    best_iou = 0.0
    best_cls = None
    for cls_id, gbox in all_gt_boxes:
        iouv = iou_xyxy(box, gbox)
        if iouv > best_iou:
            best_iou = iouv
            best_cls = cls_id
    return best_iou, best_cls

def get_strict_fp_boxes(result, gt_by_class, gt_iou_match=0.5, ignore_if_overlaps_any_gt_iou=0.3):
    tp_boxes = []
    ignored_boxes = []
    strict_fp_boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp_boxes, tp_boxes, ignored_boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    preds = [(int(pred_cls[i]), pred_xyxy[i].tolist(), float(pred_conf[i])) for i in range(len(pred_cls))]
    preds.sort(key=lambda z: z[2], reverse=True)

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}

    all_gt_boxes = []
    for gt_cls, boxes in gt_by_class.items():
        for b in boxes:
            all_gt_boxes.append((gt_cls, b))

    unmatched_preds = []

    for c, pbox, conf in preds:
        best_iou = 0.0
        best_j = None
        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= gt_iou_match:
            tp_boxes.append((c, pbox, conf, best_iou))
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            unmatched_preds.append((c, pbox, conf, best_iou))

    for c, pbox, conf, _ in unmatched_preds:
        best_any_iou, best_any_cls = best_iou_against_all_gt(pbox, all_gt_boxes)

        if best_any_iou >= ignore_if_overlaps_any_gt_iou:
            ignored_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))
        else:
            strict_fp_boxes.append((c, pbox, conf, best_any_iou, best_any_cls))

    return strict_fp_boxes, tp_boxes, ignored_boxes

def draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names):
    out = cv2.cvtColor(np.array(img_rgb), cv2.COLOR_RGB2BGR)

    for cls_id, box, conf, best_iou, best_any_cls in strict_fp_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        if isinstance(class_names, dict):
            pred_name = class_names.get(int(cls_id), str(cls_id))
        else:
            pred_name = class_names[int(cls_id)] if 0 <= int(cls_id) < len(class_names) else str(cls_id)

        label = f"FP {pred_name} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        y_text = max(18, y1)
        cv2.rectangle(out, (x1, y_text - th - 6), (x1 + tw + 4, y_text), (0, 0, 255), -1)
        cv2.putText(out, label, (x1 + 2, y_text - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

# -------------------------
# LOAD MODEL
# -------------------------
model = YOLO(MODEL_PATH)
class_names = model.names
device = next(model.model.parameters()).device

img_path = DATASET_ROOT / "images" / SPLIT / IMAGE_NAME
lab_path = DATASET_ROOT / "labels" / SPLIT / f"{Path(IMAGE_NAME).stem}.txt"

if not img_path.exists():
    raise FileNotFoundError(f"Missing image: {img_path}")

# -------------------------
# PREP IMAGE
# -------------------------
pil_img = Image.open(img_path).convert("RGB")
image_tensor = demo_transform(pil_img).to(device)
img_rgb = (image_tensor.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)

# -------------------------
# PREDICT
# -------------------------
results = model.predict(
    source=image_tensor.unsqueeze(0),
    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,
    max_det=MAX_DET,
    device=device,
    verbose=False,
)
r = results[0]

# -------------------------
# LOAD GT
# -------------------------
gt = read_yolo_gt_labels(lab_path)
H, W = IMGSZ, IMGSZ

gt_by_class = {}
for c, x, y, w, h in gt:
    gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h, W, H))

# -------------------------
# GET STRICT FPs
# -------------------------
strict_fp_boxes, tp_boxes, ignored_boxes = get_strict_fp_boxes(
    r,
    gt_by_class,
    gt_iou_match=GT_IOU_MATCH,
    ignore_if_overlaps_any_gt_iou=IGNORE_IF_OVERLAPS_ANY_GT_IOU,
)

print(
    f"{img_path.name}: "
    f"GT={len(gt)} | preds={0 if r.boxes is None else len(r.boxes)} | "
    f"TP={len(tp_boxes)} | strict_FP={len(strict_fp_boxes)} | ignored={len(ignored_boxes)}"
)

# image with only false positives
fp_vis_rgb = draw_strict_fp_boxes(img_rgb, strict_fp_boxes, class_names)

# -------------------------
# SHOW 2 IMAGES
# -------------------------
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(fp_vis_rgb)
plt.title(f"{IMAGE_NAME} - strict false positives")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_rgb)
plt.title(f"{IMAGE_NAME} - original")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# SAVE 2 IMAGES (FULL STYLE FP)
# =========================
from pathlib import Path
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
DATASET_ROOT = Path("/content/drive/MyDrive/patched/yolo_filtered")
SPLIT = "train"
IMAGE_NAME = "rgb_0067.png"

IMGSZ = 640
CONF = 0.25
IOU = 0.6

GT_IOU_MATCH = 0.5
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.3

OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(exist_ok=True)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])

def read_gt(txt):
    if not txt.exists():
        return []
    lines = txt.read_text().strip().splitlines()
    out = []
    for ln in lines:
        p = ln.split()
        if len(p) >= 5:
            out.append((int(float(p[0])), *map(float, p[1:5])))
    return out

def xywhn_to_xyxy(x,y,w,h,W,H):
    return [(x-w/2)*W,(y-h/2)*H,(x+w/2)*W,(y+h/2)*H]

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1,ix2,iy2=max(ax1,bx1),max(ay1,by1),min(ax2,bx2),min(ay2,by2)
    iw,ih=max(0,ix2-ix1),max(0,iy2-iy1)
    inter=iw*ih
    ua=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/ua if ua>0 else 0

def best_iou_all(box, gt_boxes):
    best_iou = 0.0
    best_cls = None
    for c, g in gt_boxes:
        i = iou(box, g)
        if i > best_iou:
            best_iou = i
            best_cls = c
    return best_iou, best_cls

# -------------------------
# LOAD
# -------------------------
model = YOLO(MODEL_PATH)
names = model.names
device = next(model.model.parameters()).device

img_path = DATASET_ROOT / "images" / SPLIT / IMAGE_NAME
lab_path = DATASET_ROOT / "labels" / SPLIT / f"{Path(IMAGE_NAME).stem}.txt"

pil = Image.open(img_path).convert("RGB")
tensor = transform(pil).to(device)
img_rgb = (tensor.cpu().permute(1,2,0).numpy()*255).astype(np.uint8)

# -------------------------
# PREDICT
# -------------------------
r = model.predict(tensor.unsqueeze(0), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]

# -------------------------
# GT
# -------------------------
gt = read_gt(lab_path)
gt_boxes = [(c, xywhn_to_xyxy(x,y,w,h,IMGSZ,IMGSZ)) for c,x,y,w,h in gt]

# -------------------------
# STRICT FP LOGIC
# -------------------------
strict_fp = []

if r.boxes is not None:
    preds = list(zip(
        r.boxes.cls.cpu().numpy().astype(int),
        r.boxes.xyxy.cpu().numpy(),
        r.boxes.conf.cpu().numpy()
    ))

    preds.sort(key=lambda x: x[2], reverse=True)

    for pc, pbox, conf in preds:
        best_same = max([iou(pbox, g) for c,g in gt_boxes if c==pc] or [0])
        if best_same >= GT_IOU_MATCH:
            continue

        best_any, best_cls = best_iou_all(pbox, gt_boxes)

        if best_any < IGNORE_IF_OVERLAPS_ANY_GT_IOU:
            strict_fp.append((pc, pbox, conf, best_any, best_cls))

# -------------------------
# DRAW (SAME STYLE AS YOUR CODE)
# -------------------------
fp_img = cv2.cvtColor(img_rgb.copy(), cv2.COLOR_RGB2BGR)

for cls_id, box, conf, best_iou, best_cls in strict_fp:
    x1,y1,x2,y2 = map(int, box)
    cv2.rectangle(fp_img,(x1,y1),(x2,y2),(0,0,255),2)

    pred_name = names[int(cls_id)] if int(cls_id) < len(names) else str(cls_id)
    gt_name = names[int(best_cls)] if best_cls is not None and int(best_cls) < len(names) else "none"

    label = f"FP {pred_name} {conf:.2f} IoU {best_iou:.2f} vs {gt_name}"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
    y_text = max(18, y1)

    cv2.rectangle(fp_img,(x1,y_text-th-6),(x1+tw+4,y_text),(0,0,255),-1)
    cv2.putText(fp_img,label,(x1+2,y_text-4),
                cv2.FONT_HERSHEY_SIMPLEX,0.42,(255,255,255),1,cv2.LINE_AA)

fp_img = cv2.cvtColor(fp_img, cv2.COLOR_BGR2RGB)

# -------------------------
# SAVE ONLY 2 IMAGES
# -------------------------
cv2.imwrite(str(OUT_DIR / "rgb_0067_fp.png"), cv2.cvtColor(fp_img, cv2.COLOR_RGB2BGR))
cv2.imwrite(str(OUT_DIR / "rgb_0067_clean.png"), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))

In [ ]:
# =========================
# SAVE / SHOW ALL FALSE POSITIVES FOR WHOLE SPLIT
# =========================
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt
import math
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
DATASET_ROOT = Path("/content/drive/MyDrive/patched/yolo_filtered")
SPLIT = "train"

IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000
GT_IOU_MATCH = 0.5

OUT_DIR = Path("/content/drive/MyDrive/isaac_eval_single_dataset/all_false_positives_train")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PREVIEW_K = 12  # how many saved FP images to preview at the end

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])

def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out

def xywhn_to_xyxy(x, y, w, h, W, H):
    x1 = (x - w / 2.0) * W
    y1 = (y - h / 2.0) * H
    x2 = (x + w / 2.0) * W
    y2 = (y + h / 2.0) * H
    return [x1, y1, x2, y2]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def cls_name(class_names, c):
    if isinstance(class_names, dict):
        return class_names.get(int(c), str(c))
    return class_names[int(c)] if 0 <= int(c) < len(class_names) else str(c)

def get_fp_boxes(result, gt_by_class, gt_iou_match=0.5):
    fp_boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return fp_boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    preds = [(int(pred_cls[i]), pred_xyxy[i].tolist(), float(pred_conf[i])) for i in range(len(pred_cls))]
    preds.sort(key=lambda z: z[2], reverse=True)

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}

    for c, pbox, conf in preds:
        best_iou = 0.0
        best_j = None
        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= gt_iou_match:
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            fp_boxes.append((c, pbox, conf, best_iou))

    return fp_boxes

def draw_fp_boxes(img_rgb, fp_boxes, class_names):
    out = cv2.cvtColor(np.array(img_rgb), cv2.COLOR_RGB2BGR)

    for cls_id, box, conf, best_iou in fp_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        label = f"FP {cls_name(class_names, cls_id)} {conf:.2f} IoU {best_iou:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        y_text = max(18, y1)
        cv2.rectangle(out, (x1, y_text - th - 6), (x1 + tw + 4, y_text), (0, 0, 255), -1)
        cv2.putText(out, label, (x1 + 2, y_text - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

model = YOLO(MODEL_PATH)
class_names = model.names
device = next(model.model.parameters()).device

img_dir = DATASET_ROOT / "images" / SPLIT
lab_dir = DATASET_ROOT / "labels" / SPLIT

image_paths = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}])
if not image_paths:
    raise RuntimeError(f"No images found in {img_dir}")

saved_paths = []
per_image_counts = []
total_fp = 0

for idx, orig_path in enumerate(image_paths, start=1):
    pil_img = Image.open(orig_path).convert("RGB")
    image_tensor = demo_transform(pil_img).to(device)
    H, W = IMGSZ, IMGSZ

    results = model.predict(
        source=image_tensor.unsqueeze(0),
        imgsz=IMGSZ,
        conf=CONF,
        iou=IOU,
        max_det=MAX_DET,
        device=device,
        verbose=False,
    )
    r = results[0]

    gt_path = lab_dir / f"{orig_path.stem}.txt"
    gt = read_yolo_gt_labels(gt_path)

    gt_by_class = {}
    for c, x, y, w, h in gt:
        gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h, W, H))

    fp_boxes = get_fp_boxes(r, gt_by_class, gt_iou_match=GT_IOU_MATCH)
    fp_count = len(fp_boxes)
    total_fp += fp_count
    per_image_counts.append((orig_path.name, fp_count))

    img_rgb = (image_tensor.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    fp_vis_rgb = draw_fp_boxes(img_rgb, fp_boxes, class_names)

    out_path = OUT_DIR / f"{orig_path.stem}_fp.png"
    Image.fromarray(fp_vis_rgb).save(out_path)
    saved_paths.append(out_path)

    if idx == 1 or idx % 25 == 0:
        print(f"Processed {idx}/{len(image_paths)} images | current image FP={fp_count}")

print(f"\nDone. Saved {len(saved_paths)} FP-only visualizations to:")
print(OUT_DIR)
print(f"Total FP across split: {total_fp}")
print(f"Mean FP/image: {total_fp / len(image_paths):.4f}")

# print top FP-heavy images
top_fp = sorted(per_image_counts, key=lambda x: x[1], reverse=True)[:15]
print("\nTop images by FP count:")
for name, count in top_fp:
    print(f"{name}: {count}")

# preview first K saved images
preview_paths = saved_paths[:PREVIEW_K]
cols = 2
rows = math.ceil(len(preview_paths) / cols)
plt.figure(figsize=(16, 6 * rows))

for i, p in enumerate(preview_paths, 1):
    img = Image.open(p).convert("RGB")
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(p.name, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# SAVE rgb_0250 WITH ONLY FALSE POSITIVES
# LESS STRICT VERSION
# no IoU text on image
# =========================
from pathlib import Path
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms

MODEL_PATH = "/content/drive/MyDrive/isaac_scene1/yolo_runs/yolov8n_isaac_scene12/weights/best.pt"
DATASET_ROOT = Path("/content/drive/MyDrive/patched/yolo_filtered")
SPLIT = "train"
IMAGE_NAME = "rgb_0250.png"

IMGSZ = 640
CONF = 0.25
IOU = 0.6
MAX_DET = 1000

# LESS STRICT:
GT_IOU_MATCH = 0.3
IGNORE_IF_OVERLAPS_ANY_GT_IOU = 0.15

OUT_PATH = "/content/drive/MyDrive/isaac_eval_single_dataset/rgb_0250_false_positives_only_less_strict.png"

demo_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((IMGSZ, IMGSZ)),
])

def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out

def xywhn_to_xyxy(x, y, w, h, W, H):
    x1 = (x - w / 2.0) * W
    y1 = (y - h / 2.0) * H
    x2 = (x + w / 2.0) * W
    y2 = (y + h / 2.0) * H
    return [x1, y1, x2, y2]

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def best_iou_against_all_gt(box, all_gt_boxes):
    best_iou = 0.0
    best_cls = None
    for cls_id, gbox in all_gt_boxes:
        iouv = iou_xyxy(box, gbox)
        if iouv > best_iou:
            best_iou = iouv
            best_cls = cls_id
    return best_iou, best_cls

def get_loose_fp_boxes(result, gt_by_class, gt_iou_match=0.3, ignore_if_overlaps_any_gt_iou=0.15):
    tp_boxes = []
    ignored_boxes = []
    strict_fp_boxes = []

    if result.boxes is None or len(result.boxes) == 0:
        return strict_fp_boxes, tp_boxes, ignored_boxes

    pred_cls = result.boxes.cls.detach().cpu().numpy().astype(int)
    pred_conf = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    pred_xyxy = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    preds = [(int(pred_cls[i]), pred_xyxy[i].tolist(), float(pred_conf[i])) for i in range(len(pred_cls))]
    preds.sort(key=lambda z: z[2], reverse=True)

    remaining = {c: list(lst) for c, lst in gt_by_class.items()}

    all_gt_boxes = []
    for gt_cls, boxes in gt_by_class.items():
        for b in boxes:
            all_gt_boxes.append((gt_cls, b))

    unmatched_preds = []

    for c, pbox, conf in preds:
        best_iou = 0.0
        best_j = None
        gtl = remaining.get(c, [])

        for j, gbox in enumerate(gtl):
            iouv = iou_xyxy(pbox, gbox)
            if iouv > best_iou:
                best_iou = iouv
                best_j = j

        if best_j is not None and best_iou >= gt_iou_match:
            tp_boxes.append((c, pbox, conf))
            gtl.pop(best_j)
            remaining[c] = gtl
        else:
            unmatched_preds.append((c, pbox, conf))

    for c, pbox, conf in unmatched_preds:
        best_any_iou, best_any_cls = best_iou_against_all_gt(pbox, all_gt_boxes)
        if best_any_iou >= ignore_if_overlaps_any_gt_iou:
            ignored_boxes.append((c, pbox, conf))
        else:
            strict_fp_boxes.append((c, pbox, conf))

    return strict_fp_boxes, tp_boxes, ignored_boxes

def cls_name(class_names, c):
    if isinstance(class_names, dict):
        return class_names.get(int(c), str(c))
    return class_names[int(c)] if 0 <= int(c) < len(class_names) else str(c)

def draw_fp_boxes_no_iou(img_rgb, strict_fp_boxes, class_names):
    out = cv2.cvtColor(np.array(img_rgb), cv2.COLOR_RGB2BGR)

    for cls_id, box, conf in strict_fp_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)

        label = f"FP {cls_name(class_names, cls_id)} {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        y_text = max(18, y1)
        cv2.rectangle(out, (x1, y_text - th - 6), (x1 + tw + 4, y_text), (0, 0, 255), -1)
        cv2.putText(out, label, (x1 + 2, y_text - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

model = YOLO(MODEL_PATH)
class_names = model.names
device = next(model.model.parameters()).device

img_path = DATASET_ROOT / "images" / SPLIT / IMAGE_NAME
gt_path = DATASET_ROOT / "labels" / SPLIT / f"{Path(IMAGE_NAME).stem}.txt"

if not img_path.exists():
    raise FileNotFoundError(f"Missing image: {img_path}")
if not gt_path.exists():
    raise FileNotFoundError(f"Missing label: {gt_path}")

pil_img = Image.open(img_path).convert("RGB")
image_tensor = demo_transform(pil_img).to(device)
H, W = IMGSZ, IMGSZ

results = model.predict(
    source=image_tensor.unsqueeze(0),
    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,
    max_det=MAX_DET,
    device=device,
    verbose=False,
)
r = results[0]

gt = read_yolo_gt_labels(gt_path)
gt_by_class = {}
for c, x, y, w, h in gt:
    gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h, W, H))

strict_fp_boxes, tp_boxes, ignored_boxes = get_loose_fp_boxes(
    r,
    gt_by_class,
    gt_iou_match=GT_IOU_MATCH,
    ignore_if_overlaps_any_gt_iou=IGNORE_IF_OVERLAPS_ANY_GT_IOU,
)

print(f"{IMAGE_NAME}: GT={len(gt)} | preds={0 if r.boxes is None else len(r.boxes)} | TP={len(tp_boxes)} | loose_FP={len(strict_fp_boxes)} | ignored={len(ignored_boxes)}")

img_rgb = (image_tensor.detach().cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8)
fp_vis_rgb = draw_fp_boxes_no_iou(img_rgb, strict_fp_boxes, class_names)

Image.fromarray(fp_vis_rgb).save(OUT_PATH)
print("Saved:", OUT_PATH)